# MIPY v2: full analysis of GSE65622

This is the pipeline behind the revised MIPY paper (rerun of 21 October 2026).
It needs mipy 2.1.1 or later: either `pip install mipy-impute`, or open this
notebook from the repository, where CELL 1 finds the package in `../src`.

Put the two GSE65622 analysis files in `Missing_Python\data` (they are not in
the repository; see the paper's data statement). All settings are in
**CELL 2** (`PROFILE = "complete" | "standard" | "quick"`). Finished work is
saved in `mipy_run\ckpt`, so a stopped run continues where it stopped. Every
number, table and figure the manuscript uses is written to
`mipy_run\manuscript`, `mipy_run\results` and `mipy_run\figures`; the files of
the paper's run are in `paper/output` of the repository.

### CELL 1   SETUP

In [ ]:
# ======================================================================
#  CELL 1   SETUP
#  Libraries, plot style and colours for every later cell. Run it after
#  Part 1 has said READY (or straight after a kernel restart).
# ======================================================================
import os, sys, re, json, time, pickle, hashlib, platform, importlib, warnings
from datetime import datetime
from pathlib import Path

# Part 1 ran in this session and did not finish: stop here, with the reason.
if globals().get("MIPY_READY") is False:
    raise RuntimeError("Part 1 ended with NOT READY. Scroll up to Part 1, do what "
                       "its last lines say, run it again, and start Part 2 after "
                       "it says READY.")

# Everything Part 2 needs must be installed in THIS kernel. Checked here, so
# a missing package is reported now and plainly, not halfway through Part 2.
import importlib.util
_need = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
         "sklearn": "scikit-learn", "matplotlib": "matplotlib", "jinja2": "jinja2"}
_missing = [pip for mod, pip in _need.items() if importlib.util.find_spec(mod) is None]
if _missing:
    raise ImportError(f"Not installed in the kernel {sys.executable}: "
                      f"{', '.join(_missing)}. Run the Part 1 cell at the top of "
                      "this notebook first; it installs what is missing.")
import numpy as np
import pandas as pd
import matplotlib

# Inside Jupyter figures are drawn inline; as a plain script they are only
# saved to disk.
try:
    _ip = get_ipython()
except NameError:
    _ip = None
if _ip is not None:
    try:
        _ip.run_line_magic("matplotlib", "inline")
    except Exception:
        pass                     # Jupyter's own default already draws inline
    IN_NOTEBOOK = True
else:
    matplotlib.use("Agg")
    IN_NOTEBOOK = False
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except ImportError:              # never true inside Jupyter
    def display(x):
        print(x)

    def Markdown(x):
        return x

# Expected, harmless warnings (empty slices when a protein is absent for a
# whole patient, zero-variance constant imputations) are silenced so the
# output stays readable. Real errors still raise.
warnings.filterwarnings("ignore")
np.seterr(invalid="ignore", divide="ignore")

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 80)

# Plot style. Set one at a time, so a setting that some matplotlib version
# does not know is skipped instead of stopping the cell.
for _k, _v in {"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
               "axes.spines.top": False, "axes.spines.right": False,
               "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
               "legend.frameon": False,
               "figure.constrained_layout.use": True}.items():
    try:
        plt.rcParams[_k] = _v
    except (KeyError, ValueError):
        pass

# Okabe-Ito, colour-blind safe. Observed is always blue, imputed orange.
OKABE = {"obs": "#0072B2", "imp": "#E69F00", "third": "#009E73",
         "grey": "#666666", "red": "#D55E00"}
MECH_COLOURS = [OKABE["obs"], OKABE["imp"], OKABE["third"]]

# Which environment is this? Anaconda base runs the core methods only.
_prefix = Path(sys.prefix)
_env = _prefix.name if _prefix.parent.name.lower() == "envs" else (
    "base" if (_prefix / "conda-meta").is_dir() else _prefix.name)
if _env == "base":
    print("  Note: this kernel is Anaconda base, so miceforest, HyperImpute, GAIN\n"
          "  and the survival stage will be skipped. For all methods use\n"
          "  Kernel -> Change kernel -> Python (gse).")

# ---- the mipy package (MIPY v2) ------------------------------------------
# Looked for in this order: already installed, then the folder mipy_pkg next
# to the notebook's data (unzip MIPY_v2_package.zip there). Added to the
# search path so this kernel and its worker processes can import it.
_MIPY_SRC = [Path(os.environ.get("MIPY_PKG", "")) / "src",
             Path(os.environ.get("MIPY_WORK_DIR", Path.home() / "Missing_Python")) / "mipy_pkg" / "src",
             Path.cwd() / "mipy_pkg" / "src",
             Path.cwd().parent / "src"]                 # notebooks/ of the repository
for _p in _MIPY_SRC:
    if (_p / "mipy" / "__init__.py").is_file() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
        break
try:
    import mipy
except ImportError:
    raise ImportError("The mipy package was not found. Unzip MIPY_v2_package.zip into "
                      f"{Path.home() / 'Missing_Python'} so that the folder mipy_pkg "
                      "is there, then run this cell again.")
if tuple(int(x) for x in mipy.__version__.split(".")[:3]) < (2, 1, 1):
    raise ImportError(f"mipy {mipy.__version__} is too old; this pipeline needs 2.1.1 or "
                      "later (probit design, effective sample sizes)")

T0 = time.time()
print(f"CELL 1 OK   kernel {_env}   python {sys.version.split()[0]}   "
      f"numpy {np.__version__}   pandas {pd.__version__}   "
      f"{'Jupyter (inline figures)' if IN_NOTEBOOK else 'script mode'}   mipy {mipy.__version__}")

### CELL 2   CONFIGURATION   <-- the only cell you edit

In [ ]:
# ======================================================================
#  CELL 2   CONFIGURATION   <-- the only cell you edit
#
#  PROFILE sets how much of the extended (v2) analysis runs:
#    "complete"  everything: all methods repeated over 10 maskings, multiple
#                imputation for every MI-capable method, R agreement, data
#                handling up to 10 million patients, the synthetic second
#                cohort. About 8-12 hours with 4 worker processes; it can be
#                stopped and restarted at any time (finished work is saved).
#    "standard"  slow methods repeated 3 times, slow MI variants skipped,
#                data handling up to 1 million patients. About 3-4 hours.
#    "quick"     a few minutes, to check that everything runs.
# ======================================================================
PROFILE = "complete"
SEED = 2026                       # one seed for every random step

# Part 1 records where the data is. Part 2 reads that record first, so the
# files are found even after a kernel restart.
WORK_DIR = Path(os.environ.get("MIPY_WORK_DIR", Path.home() / "Missing_Python"))
SETUP_FILE = WORK_DIR / "mipy_setup.json"
try:
    SETUP = json.loads(SETUP_FILE.read_text(encoding="utf-8"))
except Exception:
    SETUP = {}

# Where the two CSVs live, if Part 1's record is missing. Searched in order.
DATA_DIRS = [
    Path("N:/Research/MIPY/Data"),
    Path.home() / "Research" / "MIPY" / "Data",
    Path.cwd() / "Data",
    Path.cwd(),
]
if SETUP.get("data_dir"):
    DATA_DIRS.insert(0, Path(SETUP["data_dir"]))
DATASETS = {                      # label -> file patterns, survival-time column
    "RFS": {"files": ["RFS_data*.csv"], "surv": ["RFS"]},   # relapse-free survival
    "MFS": {"files": ["MFS_data*.csv"], "surv": ["MFS"]},   # metastasis-free survival
}
EVENT_COL = "event"

# Where results go. Keep this on the local disk, not N:.
OUT_DIR = WORK_DIR / "mipy_run"

QUICK = False     # True  -> 6 fast methods, 2 sweeps: a 2-minute trial run
FRESH = False     # True  -> ignore saved imputations and recompute everything

# ---- data preparation -------------------------------------------------
LOG_TRANSFORM = True              # log2(x + 1); see manuscript section 4.2
SELECT_BY = "all"                 # "all" | "cv" | "observed"
N_PROTEINS_SEL = 507              # used only when SELECT_BY != "all"
MAX_PCT_MISSING = 90.0            # drop proteins missing more than this

# ---- amputation (ground truth) -----------------------------------------
AMPUTE_RATE = 0.20                # fraction of OBSERVED cells masked
MECHANISMS = ("MCAR", "MAR", "MNAR")
AMP_STRENGTH = 2.0                # logistic slope (beta) for MAR and MNAR
MIN_OBS_PER_PROTEIN = 12          # a protein keeps at least this many cells

# ---- imputation --------------------------------------------------------
MAX_PREDICTORS = 25               # quickpred cap (van Buuren 2018, 9.1.6)
MIN_COR = 0.10                    # quickpred admission threshold
MICE_ITER = 5                     # chained-equation sweeps
N_IMPUTATIONS = 5                 # m, for the multiple-imputation method
KNN_K = 15                        # Troyanskaya et al. 2001: k in 10-20
SOFT_RANK = 30                    # softImpute maximum rank
GAIN_EPOCHS = 100                 # NOTE: has no effect. hyperimpute's GAIN plugin
                                  # (0.1.10 to 0.1.17) does not pass n_epochs, batch
                                  # size or hint rate to the model it trains, which
                                  # uses its own defaults (1,000 iterations, batch 256,
                                  # hint rate 0.9). CELL 4 records what was used. The
                                  # value stays at 100 because it is part of every
                                  # saved run's fingerprint.
HYPER_INNER_ITER = 3              # HyperImpute inner iterations
N_JOBS = -1

N_WORKERS = mipy.runner.default_workers(4)   # parallel worker processes (1 = none)
MNAR_GAMMA = 1.0                  # MICE-LongMNAR: assumed censoring strength
GAMMA_GRID = (0.0, 0.5, 1.0, 2.0, 3.0)        # its sensitivity analysis
MI_METHODS = ("MICE-Bayes", "MICE-Long", "MICE-LongMNAR",   # multiple imputation
              "softImpute-MI", "missForest-MI", "miceforest-MI")
MI_SLOW = True                    # include missForest-MI and miceforest-MI

# ---- robustness: repeated masking (Stage 4.4) --------------------------
N_REPEATS = 10                    # independent maskings; 1 = the main run only
REPEAT_SLOW = True                # also repeat missForest, miceforest,
                                  # HyperImpute and GAIN
N_REPEATS_SLOW = 10               # maskings for those four
SLOW_METHODS = ("missForest", "miceforest", "HyperImpute", "GAIN")

# ---- MNAR sensitivity (Stage 4.6) --------------------------------------
SENS_STRENGTHS = (1.0, 4.0)       # extra MNAR slopes (2.0 is the main run)
RUN_LOD = True                    # hard detection-limit masking
RUN_PROBIT = True                 # not-at-random masking with a probit selection,
PROBIT_SLOPE = "matched"          # Phi(alpha - beta z); "matched" = the slope whose
                                  # masked cells have the same expected mean z as the
                                  # logistic MNAR design with AMP_STRENGTH (or a number)
RUN_ESS = True                    # effective sample size of MICE-LongMNAR's importance
                                  # resampling, for every gamma > 0 on every design

# ---- R agreement (Stage 4.8) -------------------------------------------
RUN_R = True                      # needs R; the R packages install themselves
RSCRIPT = None                    # path to Rscript.exe if not found automatically
R_MF_NTREE, R_MF_MAXITER = 100, 10

# ---- data handling at scale (Stage 4.9) --------------------------------
RUN_BIGDATA = True
BIG_SIZES = (10_000, 100_000, 1_000_000, 10_000_000)   # patients
BIG_FEATURES = 10
BIG_STEP_LIMIT = 900              # seconds per step before larger sizes are skipped

# ---- second cohort: calibrated synthetic data (Stage 4.10) -------------
RUN_SIM = True
SIM_PATIENTS, SIM_PROTEINS = 200, 200
SIM_REPEATS = 3
SIM_SLOW = False                  # also run the four slow methods on it

# ---- scaling experiment on simulated data (Stage 4.5) -------------------
RUN_SCALING = True
SCALING_SIZES = (500, 1_000, 2_000, 5_000, 10_000, 100_000)   # rows
SCALING_P = 100                   # proteins in the simulated matrix
SCALE_TIME_LIMIT = 300            # seconds; a method predicted to exceed it is skipped

# ---- survival (Stage 5) ------------------------------------------------
# "off"       : no survival stage, and no endpoint checks (the manuscript
#               revision of 19 October 2026 leaves survival out). Default.
# "reconciled": one progression endpoint built to be consistent (any relapse
#               or metastasis, earliest time); exploratory until the source
#               records are checked. The output and numbers.tex say so.
# "gated"     : only if the RFS/MFS checks pass (they do not, so it is withheld)
# "MFS_only"  : MFS alone, dropping patients with time <= 0 (exploratory)
SURVIVAL_MODE = "off"
SURV_FDR = 0.10                   # false-discovery rate for "significant" proteins

# ---- external validation targets (published R pipeline) ----------------
EXPECT = {"n_obs": 288, "n_patients": 80, "n_proteins": 507,
          "overall_missing": 10.5,
          "top_missing": {"Leptin R": 81.9, "Glut2": 77.8, "GDF-15": 73.3}}
TOL_PCT = 1.0
FORCE_PROTEIN = ["EN-RAGE", "RAGE"]   # proteins, whatever their names suggest

# Environment variables override the flags above. Used only for automated
# testing; ignore them when working interactively.
if "MIPY_QUICK" in os.environ:
    QUICK = os.environ["MIPY_QUICK"] == "1"
if "MIPY_FRESH" in os.environ:
    FRESH = os.environ["MIPY_FRESH"] == "1"
if "MIPY_OUT" in os.environ:
    OUT_DIR = Path(os.environ["MIPY_OUT"])
if "MIPY_SURVIVAL" in os.environ:
    SURVIVAL_MODE = os.environ["MIPY_SURVIVAL"]
if "MIPY_REPEATS" in os.environ:
    N_REPEATS = int(os.environ["MIPY_REPEATS"])
if "MIPY_SCALING" in os.environ:
    RUN_SCALING = os.environ["MIPY_SCALING"] == "1"
if "MIPY_SCALE_SIZES" in os.environ:
    SCALING_SIZES = tuple(int(v) for v in os.environ["MIPY_SCALE_SIZES"].split(","))
if "MIPY_PROFILE" in os.environ:
    PROFILE = os.environ["MIPY_PROFILE"]
if "MIPY_WORKERS" in os.environ:
    N_WORKERS = int(os.environ["MIPY_WORKERS"])
if "MIPY_R_NTREE" in os.environ:
    R_MF_NTREE = int(os.environ["MIPY_R_NTREE"])
if "MIPY_BIG_SIZES" in os.environ:
    BIG_SIZES = tuple(int(v) for v in os.environ["MIPY_BIG_SIZES"].split(","))
if PROFILE not in ("complete", "standard", "quick"):
    raise ValueError(f'PROFILE must be "complete", "standard" or "quick", not {PROFILE!r}')
if PROFILE == "standard":
    N_REPEATS_SLOW, MI_SLOW = 3, False
    BIG_SIZES = tuple(s for s in BIG_SIZES if s <= 1_000_000)
if PROFILE == "quick":
    QUICK = True
    N_REPEATS_SLOW, MI_SLOW, SIM_REPEATS = 2, False, 1
    BIG_SIZES = tuple(s for s in BIG_SIZES if s <= 100_000)
    SENS_STRENGTHS = SENS_STRENGTHS[:1]
    GAMMA_GRID = (0.0, 1.0, 2.0)
    SIM_PATIENTS, SIM_PROTEINS = 60, 60
    R_MF_NTREE, R_MF_MAXITER = 20, 2
if not MI_SLOW:
    MI_METHODS = tuple(m for m in MI_METHODS if m not in ("missForest-MI", "miceforest-MI"))
if QUICK:
    MICE_ITER, N_IMPUTATIONS = 2, 2
    N_REPEATS = min(N_REPEATS, 3)
    N_REPEATS_SLOW = min(N_REPEATS_SLOW, N_REPEATS)
    SCALING_SIZES = tuple(s for s in SCALING_SIZES if s <= 10_000)

if SURVIVAL_MODE not in ("off", "reconciled", "gated", "MFS_only"):
    raise ValueError(f'SURVIVAL_MODE must be "off", "reconciled", "gated" or "MFS_only", '
                     f'not {SURVIVAL_MODE!r}')
if not (PROBIT_SLOPE == "matched" or isinstance(PROBIT_SLOPE, (int, float))):
    raise ValueError(f'PROBIT_SLOPE must be "matched" or a number, not {PROBIT_SLOPE!r}')
if SELECT_BY not in ("all", "cv", "observed"):
    raise ValueError(f'SELECT_BY must be "all", "cv" or "observed", not {SELECT_BY!r}')

print(f"CELL 2 OK   output -> {OUT_DIR}")
print(f"            mode   -> profile {PROFILE}, {N_WORKERS} worker process(es)"
      f"{', QUICK trial settings' if QUICK else ''}")
print(f"            run    -> {'full' if not QUICK else 'quick'}"
      f"{', recomputing everything' if FRESH else ''}")
print(f"            data   -> "
      f"{'recorded by Part 1 in ' + str(SETUP_FILE) if SETUP.get('files') else 'not recorded yet: run Part 1 first'}")

### CELL 3   HELPERS AND OUTPUT FOLDERS

In [ ]:
# ======================================================================
#  CELL 3   HELPERS AND OUTPUT FOLDERS
#  Small functions every later cell uses: recording checks and manuscript
#  numbers, saving/loading finished imputations, and showing + saving
#  every table and figure.
# ======================================================================
DIRS = {k: OUT_DIR / k for k in
        ("figures", "results", "ckpt", "manuscript", "provenance")}
for _d in DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)

# Figures the manuscript no longer uses (revision of 19 October 2026: they
# repeated a table or left out the two new methods; fig15 belonged to the
# survival screen). They are not drawn any more, and copies from earlier
# runs are removed so they cannot be copied into Overleaf by mistake.
DROPPED_FIGURES = ("fig03_nrmse", "fig07_peer", "fig08_mi_coverage", "fig09_repeats",
                   "fig14_bigdata", "fig15_survival_overlap")
for _f in DROPPED_FIGURES:
    for _ext in (".pdf", ".png"):
        (DIRS["figures"] / f"{_f}{_ext}").unlink(missing_ok=True)

CHECKS = []        # every integrity check, written to provenance/checks.json
NUMBERS = {}       # every number the manuscript quotes, written to numbers.tex


def check(cid, gate, label, passed, detail=""):
    """Record and print one check. gate "A" = protein block, "B" = endpoints."""
    passed = bool(passed)
    CHECKS[:] = [c for c in CHECKS if c["id"] != cid]     # re-running replaces
    CHECKS.append({"id": cid, "gate": gate, "label": label,
                   "passed": passed, "detail": str(detail)})
    print(f"  [{'PASS' if passed else 'FAIL'}] {cid:<8} {label}")
    if detail:
        print(f"             {detail}")
    return passed


def num(key, value, fmt=None, raw=False):
    """Record a number for the manuscript (written to numbers.tex)."""
    NUMBERS[key] = {"value": value, "fmt": fmt, "raw": raw}


def toks(name):
    """Lower-case word tokens of a column name: 'EN-RAGE' -> ['en', 'rage']."""
    return [t for t in re.split(r"[^a-z0-9]+", str(name).lower()) if t]


def norm(name):
    """Column name with case, spaces and punctuation removed."""
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def texsafe(s):
    return (str(s).replace("\\", r"\textbackslash{}").replace("_", r"\_")
            .replace("&", r"\&").replace("%", r"\%").replace("#", r"\#")
            .replace("$", r"\$"))


def fingerprint(X, name, settings):
    """Short hash of the data, the method and every setting that changes
    its result. A saved imputation is reused only if all three match."""
    h = hashlib.sha256()
    a = np.ascontiguousarray(np.asarray(X, dtype=float))
    h.update(str(a.shape).encode())
    h.update(np.nan_to_num(a, nan=-1.2345e300).tobytes())
    h.update(str(name).encode())
    h.update(repr(tuple(settings)).encode())
    return h.hexdigest()[:16]


def save_ckpt(name, obj):
    """Save a finished result at once, so a stopped run can resume."""
    tmp = DIRS["ckpt"] / f"{name}.tmp"
    with open(tmp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, DIRS["ckpt"] / f"{name}.pkl")   # never leaves half a file


def load_ckpt(name):
    """A saved result, or None (also None when FRESH = True)."""
    p = DIRS["ckpt"] / f"{name}.pkl"
    if FRESH or not p.is_file():
        return None
    try:
        with open(p, "rb") as f:
            return pickle.load(f)
    except Exception:              # damaged file: recompute instead of failing
        return None


def show_fig(fig, name):
    """Save a figure as PDF (manuscript) and PNG, show it inline, close it."""
    fig.savefig(DIRS["figures"] / f"{name}.pdf")
    fig.savefig(DIRS["figures"] / f"{name}.png", dpi=150)
    if IN_NOTEBOOK:
        display(fig)
    plt.close(fig)


def show_table(df, title, index=True, csv=None, precision=3, gradient=None):
    """Show a table under a bold title and, if csv is given, save it."""
    if csv:
        df.to_csv(DIRS["results"] / csv, index=index)
    if not IN_NOTEBOOK:
        print(f"\n{title}\n{df.to_string(index=index)}")
        return
    display(Markdown(f"**{title}**"))
    sty = df.style.format(precision=precision, na_rep="n/a")
    if gradient:
        cols = [c for c in gradient if c in df.columns]
        try:                       # colour scale needs matplotlib; optional
            if cols:
                sty = sty.background_gradient(cmap="Blues_r", subset=cols)
        except Exception:
            pass
    if not index:
        sty = sty.hide(axis="index")
    display(sty)


print(f"CELL 3 OK   folders ready under {OUT_DIR}")

### CELL 4   STAGE 1.1   ENVIRONMENT AND FILES

In [ ]:
# ======================================================================
#  CELL 4   STAGE 1.1   ENVIRONMENT AND FILES
#  Records every package version, then finds the two data files: first
#  from Part 1's record, otherwise by searching DATA_DIRS (CELL 2).
# ======================================================================
ENV = {"python": sys.version.split()[0],
       "platform": f"{platform.system()} {platform.release()} ({platform.machine()})",
       "executable": sys.executable, "seed": SEED,
       "run_at": datetime.now().isoformat(timespec="seconds")}

# Part 1 test-ran the optional packages in a separate Python. One that
# failed there (PyTorch refusing to load, say) is not imported here at all,
# so it cannot stop or crash this kernel; its methods show as skipped.
_UNUSABLE = set()
if os.path.normcase(str(SETUP.get("python", ""))) == os.path.normcase(sys.executable):
    _UNUSABLE = {m for m, ok in (SETUP.get("usable") or {}).items() if ok is False}
# Optional packages for the data-handling stage: installed on first use
# (pip, into this kernel) if missing. A failure only skips that engine.
if RUN_BIGDATA:
    import subprocess
    for _m in ("polars", "duckdb"):
        if importlib.util.find_spec(_m) is None:
            print(f"  installing {_m} for the data-handling stage ...", flush=True)
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", _m],
                           capture_output=True, text=True)
    importlib.invalidate_caches()

import importlib.metadata as _ilm
_DIST = {"sklearn": "scikit-learn"}       # import name -> distribution name
for _m in ["numpy", "pandas", "scipy", "sklearn", "matplotlib", "statsmodels",
           "lightgbm", "miceforest", "hyperimpute", "lifelines", "polars", "duckdb",
           "psutil", "mipy"]:
    if _m in _UNUSABLE:
        ENV[_m] = None
        continue
    try:
        _mod = importlib.import_module(_m)
    except Exception:
        ENV[_m] = None
        continue
    # the installed version from the package metadata (hyperimpute, for one,
    # has no __version__); for mipy the version of the library imported here
    _v = getattr(_mod, "__version__", None) if _m == "mipy" else None
    if _v is None:
        try:
            _v = _ilm.version(_DIST.get(_m, _m))
        except Exception:
            _v = getattr(_mod, "__version__", "installed")
    ENV[_m] = _v
try:                                          # PyTorch runs GAIN and HyperImpute
    ENV["torch"] = _ilm.version("torch") if ENV.get("hyperimpute") else None
except Exception:
    ENV["torch"] = None

# What GAIN actually trains with. hyperimpute's GAIN plugin keeps the
# settings it is given but builds its model with the model's own defaults,
# so the n_epochs passed by mipy (GAIN_EPOCHS) may not be the one used. The
# model is inspected here (no training) and its settings are recorded.
GAIN_USED = None
if ENV.get("hyperimpute"):
    import random as _random
    _rng_state = (np.random.get_state(), _random.getstate())   # the plugin re-seeds globally
    try:
        from hyperimpute.plugins.imputers import Imputers as _Imputers
        _gp = _Imputers().get("gain", n_epochs=GAIN_EPOCHS, random_state=SEED)
        _gm = getattr(_gp, "_model", _gp)
        GAIN_USED = {k: getattr(_gm, k, None)
                     for k in ("n_epochs", "batch_size", "hint_rate", "loss_alpha")}
        GAIN_USED["passed_n_epochs"] = GAIN_EPOCHS
    except Exception as _e:
        print(f"  could not inspect hyperimpute's GAIN model ({type(_e).__name__}: {_e})")
    finally:
        np.random.set_state(_rng_state[0])
        _random.setstate(_rng_state[1])
if GAIN_USED:
    ENV["gain_settings_used"] = GAIN_USED
    print(f"  GAIN trains with {GAIN_USED['n_epochs']} iterations, batch size "
          f"{GAIN_USED['batch_size']}, hint rate {GAIN_USED['hint_rate']} "
          f"(mipy passes n_epochs = {GAIN_EPOCHS}"
          + ("; the plugin does not pass it on)" if GAIN_USED["n_epochs"] != GAIN_EPOCHS else ")"))
    num("GainEpochsUsed", f"{int(GAIN_USED['n_epochs']):,}")
    num("GainBatchUsed", GAIN_USED["batch_size"])
    num("GainHintUsed", GAIN_USED["hint_rate"])
    num("GainEpochsPassed", GAIN_EPOCHS)
if _UNUSABLE:
    print(f"  not used, because they failed Part 1's test run: "
          f"{', '.join(sorted(_UNUSABLE))}")

check("C1", "A", "core scientific stack importable",
      all(ENV[m] for m in ["numpy", "pandas", "scipy", "sklearn"]))

# 1. the files Part 1 recorded (copies in Missing_Python\data)
FOUND = {}
_ALIAS = {"EFS": "RFS"}                # Part 1 labels the relapse file "EFS"
_SOURCES = {_ALIAS.get(k, k): v for k, v in (SETUP.get("sources") or {}).items()}
for _lab, _f in (SETUP.get("files") or {}).items():
    _lab = _ALIAS.get(_lab, _lab)
    if _lab in DATASETS and _f and Path(_f).is_file():
        FOUND[_lab] = Path(_f)
# 2. anything still missing: search the folders listed in CELL 2
for _d in DATA_DIRS:
    try:
        if not _d.is_dir():
            continue
    except OSError:                    # e.g. N: not connected
        continue
    for _lab, _spec in DATASETS.items():
        if _lab in FOUND:
            continue
        for _pat in _spec["files"]:
            _hits = sorted(_d.glob(_pat))
            if _hits:
                FOUND[_lab] = _hits[0]
                break

if not check("C2", "A", "both datasets located", len(FOUND) == len(DATASETS),
             "searched: " + "; ".join(str(d) for d in DATA_DIRS)):
    raise FileNotFoundError("The data files were not found. Run the Part 1 "
                            "cell at the top; it finds them or lets you pick "
                            "or upload them. Then run CELLs 1 to 4 again.")

_env_tab = pd.DataFrame(
    [(k, "missing" if v is None else str(v)) for k, v in ENV.items()],
    columns=["item", "value"])
show_table(_env_tab, "Environment", index=False)
show_table(pd.DataFrame([(k, str(v), _SOURCES.get(k, str(v)))
                         for k, v in FOUND.items()],
                        columns=["dataset", "file used", "original source"]),
           "Input files", index=False)
RSCRIPT_PATH = mipy.rbridge.find_rscript(RSCRIPT) if RUN_R else None
ENV["R"] = RSCRIPT_PATH
if RUN_R and RSCRIPT_PATH is None:
    print("  R was not found, so the R-agreement stage will be skipped "
          "(set RSCRIPT in CELL 2 to the path of Rscript.exe)")
_skipped = [m for m in ("miceforest", "hyperimpute", "polars", "duckdb")
            if ENV.get(m) is None]
if _skipped:
    print(f"  not available in this kernel: {', '.join(_skipped)} "
          "(their methods will show as skipped; everything else runs)")
print("CELL 4 OK")

### CELL 5   STAGE 1.2   AUDIT EACH DATASET

In [ ]:
# ======================================================================
#  CELL 5   STAGE 1.2   AUDIT EACH DATASET
#  Classify every column, check the visit keys, validate missingness
#  against the published R pipeline.
# ======================================================================
ID_HINTS = ["patient_id", "patientid", "subject", "patient", "id"]
TIME_HINTS = ["sampling_point", "samplingpoint", "visit", "timepoint",
              "time_point", "cycle", "week", "day", "point", "occasion"]
CLINICAL_TOKENS = {
    "status", "event", "censor", "death", "surv", "efs", "rfs", "mfs", "os",
    "stage", "grade", "sex", "gender", "arm", "group", "response",
    "time", "age", "bmi", "treat", "cohort", "batch", "score", "site",
    "ypt", "ypn", "trg", "slide", "print", "run"}


def classify(df, surv_names):
    """Split columns into ID, visit, proteins and clinical covariates.

    Names are matched on whole tokens, so 'age' matches Age but not EN-RAGE.
    A float column with many distinct values is never demoted on its name.
    """
    cols = list(df.columns)
    tk = {c: toks(c) for c in cols}
    surv_norm = {norm(s) for s in surv_names} | {norm(EVENT_COL)}

    def pick(hints):
        for h in hints:                      # exact name first
            for c in cols:
                if norm(c) == norm(h):
                    return c
        for h in hints:                      # then a name containing the hint
            for c in cols:
                if norm(h) in norm(c):
                    return c
        return None

    id_col, time_col = pick(ID_HINTS), pick(TIME_HINTS)
    fp = {norm(x) for x in FORCE_PROTEIN}
    proteins, clinical = [], []
    for c in cols:
        if c in (id_col, time_col):
            continue
        if norm(c) in fp:
            proteins.append(c)
            continue
        s = df[c]
        n = int(s.nunique(dropna=True))
        if norm(c) in surv_norm:
            clinical.append((c, n, str(s.dtype), "survival"))
        elif not pd.api.types.is_numeric_dtype(s):
            clinical.append((c, n, str(s.dtype), "non-numeric"))
        elif n <= 10:
            clinical.append((c, n, str(s.dtype), "n_unique <= 10"))
        elif (CLINICAL_TOKENS & set(tk[c])) and (
                not pd.api.types.is_float_dtype(s) or n <= 30):
            clinical.append((c, n, str(s.dtype), "clinical name"))
        else:
            proteins.append(c)
    return id_col, time_col, proteins, clinical


STORE = {}
_val_rows, _sum_rows, _held_rows = [], [], []
for _lab, _path in FOUND.items():
    print(f"\n--- {_lab}  ({_path.name})")
    _sha = hashlib.sha256(_path.read_bytes()).hexdigest()
    try:
        _df = pd.read_csv(_path)
    except UnicodeDecodeError:               # a file saved by Excel in ANSI
        _df = pd.read_csv(_path, encoding="latin-1")
    _df.columns = [str(c).strip() for c in _df.columns]
    _spec = DATASETS[_lab]
    _surv = next((c for c in _df.columns
                  if norm(c) in {norm(s) for s in _spec["surv"]}), None)
    _event = next((c for c in _df.columns if norm(c) == norm(EVENT_COL)), None)
    _id, _t, _prot, _clin = classify(_df, _spec["surv"])

    check(f"{_lab}.C3", "A", "ID and visit columns identified",
          _id is not None and _t is not None, f"ID={_id}, visit={_t}")
    if _id is None or _t is None:
        raise ValueError(f"{_lab}: could not find the patient ID or visit column. "
                         f"Columns start: {list(_df.columns)[:6]}")
    _d = len(_prot) - EXPECT["n_proteins"]
    check(f"{_lab}.C4", "A", f"protein count = {EXPECT['n_proteins']}", _d == 0,
          f"difference {_d:+d}" if _d else "")
    check(f"{_lab}.C5", "A", "no duplicate (patient, visit) rows",
          int(_df.duplicated([_id, _t]).sum()) == 0)
    check(f"{_lab}.C6", "A", f"{EXPECT['n_obs']} observations",
          len(_df) == EXPECT["n_obs"], f"got {len(_df)}")
    check(f"{_lab}.C7", "A", f"{EXPECT['n_patients']} patients",
          _df[_id].nunique() == EXPECT["n_patients"],
          f"got {_df[_id].nunique()}")
    if SURVIVAL_MODE != "off":                  # endpoint check, only with survival
        check(f"{_lab}.C7b", "B", "survival time and event columns present",
              _surv is not None and _event is not None,
              f"time={_surv}, event={_event}")

    # proteins must be numeric; anything else (e.g. text like 'NA ') -> missing
    _P = _df[_prot].apply(pd.to_numeric, errors="coerce")
    _df[_prot] = _P
    _overall = float(_P.isna().to_numpy(copy=True).mean() * 100)
    _pmiss = (_P.isna().mean() * 100).sort_values(ascending=False)
    _ok_overall = abs(_overall - EXPECT["overall_missing"]) <= TOL_PCT
    check(f"{_lab}.C8", "A",
          f"overall missingness matches R ({EXPECT['overall_missing']}%)",
          _ok_overall, f"got {_overall:.2f}%")
    _val_rows.append((_lab, "overall missing (%)", EXPECT["overall_missing"],
                      round(_overall, 2), _ok_overall))

    # the three named proteins published by the R pipeline
    _lut = {norm(c): c for c in _prot}
    _ok_all = True
    for _nm, _ev in EXPECT["top_missing"].items():
        _c = _lut.get(norm(_nm)) or next((v for k, v in _lut.items()
                                          if norm(_nm) in k), None)
        _got = None if _c is None else round(float(_pmiss[_c]), 1)
        _ok = _got is not None and abs(_got - _ev) <= TOL_PCT
        _ok_all &= _ok
        _val_rows.append((_lab, f"{_nm} missing (%)", _ev, _got, _ok))
    check(f"{_lab}.C9", "A", "named proteins match published missingness", _ok_all)

    _full = int((_P.notna().all()).sum())
    STORE[_lab] = {"df": _df, "path": _path, "sha256": _sha, "id": _id,
                   "time": _t, "proteins": _prot, "clinical": _clin,
                   "surv": _surv, "event": _event, "pmiss": _pmiss,
                   "overall_missing": _overall, "fully_observed": _full}
    _sum_rows.append((_lab, len(_df), _df[_id].nunique(), _df.shape[1],
                      len(_prot), len(_clin), round(_overall, 2), _full,
                      _sha[:16] + "..."))
    _held_rows += [(_lab, c, n, dt, why) for c, n, dt, why in _clin]

show_table(pd.DataFrame(_sum_rows, columns=[
    "dataset", "rows", "patients", "columns", "proteins", "clinical",
    "missing %", "fully observed proteins", "sha256"]),
    "Dataset summary", index=False, csv="s1_dataset_summary.csv")
show_table(pd.DataFrame(_held_rows, columns=[
    "dataset", "column", "distinct values", "dtype", "why not a protein"]),
    "Columns held out of the protein block", index=False,
    csv="s1_heldout_columns.csv")
show_table(pd.DataFrame(_val_rows, columns=[
    "dataset", "quantity", "published (R)", "Python", "match"]),
    "External validation against the published R pipeline", index=False,
    csv="s1_external_validation.csv")

# Reference dataset for everything that follows: the protein block is the
# same in both files (CELL 7 verifies it), so the first one is used.
_A = list(STORE)[0]
REF = STORE[_A]
_top = REF["pmiss"].head(20).round(1).to_frame("missing %")
show_table(_top, f"Top 20 proteins by missingness ({_A})", precision=1,
           csv="s1_top20_missing.csv")

num("NObs", len(REF["df"]))
num("NPatients", int(REF["df"][REF["id"]].nunique()))
num("NProteins", len(REF["proteins"]))
num("NColumns", REF["df"].shape[1])
num("PctMissing", round(REF["overall_missing"], 2), ".2f")
num("NFullyObserved", REF["fully_observed"])
print("CELL 5 OK")

### CELL 6   FIGURE 2 OF THE MANUSCRIPT   MISSINGNESS PER PROTEIN

In [ ]:
# ======================================================================
#  CELL 6   FIGURE 2 OF THE MANUSCRIPT   MISSINGNESS PER PROTEIN
#  The 507 proteins ranked by their share of missing samples (the full
#  distribution, value by value); dashed line at 50%, the three most
#  incomplete proteins labelled. (The histogram panel of earlier versions
#  was dropped in the revision of 19 October 2026: the ranked curve already
#  shows the whole distribution, and a histogram hides individual values.)
#  The per-protein values are saved in results/s1_missingness_per_protein.csv.
# ======================================================================
_pm = REF["pmiss"].sort_values(ascending=False)          # % missing per protein
_n50 = int((_pm > 50).sum())
_pm.rename("percent missing").rename_axis("protein").to_csv(
    DIRS["results"] / "s1_missingness_per_protein.csv")

fig, axB = plt.subplots(1, 1, figsize=(5.6, 3.4))
axB.plot(np.arange(1, len(_pm) + 1), _pm.to_numpy(), color=OKABE["obs"], lw=1.2)
axB.axhline(50, ls="--", lw=0.8, color=OKABE["grey"])
for _i, (_name, _v) in enumerate(_pm.head(3).items()):  # label the top three
    axB.annotate(f"{_name} ({_v:.1f}%)", (_i + 1, _v),
                 xytext=(25, -6 - 11 * _i), textcoords="offset points",
                 fontsize=7, color=OKABE["grey"],
                 arrowprops=dict(arrowstyle="-", lw=0.5, color=OKABE["grey"]))
axB.set_xlabel("protein rank")
axB.set_ylabel("samples with the protein missing (%)")
axB.set_ylim(0, 100)
axB.set_xlim(0, len(_pm) + 1)
show_fig(fig, "fig01_missingness")

_desc = pd.DataFrame({
    "quantity": ["proteins", "with no missing value", "under 5% missing",
                 "over 50% missing", "median % missing", "maximum % missing"],
    "value": [len(_pm), int((_pm == 0).sum()), int((_pm < 5).sum()), _n50,
              round(float(_pm.median()), 1), round(float(_pm.max()), 1)]})
show_table(_desc, "Figure 2 in numbers", index=False, precision=1,
           csv="s1_missingness_summary.csv")
num("NOverHalfMissing", _n50)
num("MedianPctMissing", round(float(_pm.median()), 1), ".1f")
print("CELL 6 OK   figure saved as figures/fig01_missingness.pdf and .png "
      "(Figure 2 of the manuscript)")

### CELL 7   STAGE 1.3   CROSS-DATASET COMPARISON AND VISIT STRUCTURE

In [ ]:
# ======================================================================
#  CELL 7   STAGE 1.3   CROSS-DATASET COMPARISON AND VISIT STRUCTURE
#  (a) the protein blocks of the two files must be identical, so the
#      imputation can run once and serve both endpoints;
#  (b) which visits each patient has, and whether the lost visits and
#      the missing protein values depend on the sampling point.
# ======================================================================
from scipy import stats as _stats

# ---- (a) the two protein blocks ----------------------------------------
_labs = list(STORE)
if len(_labs) == 2:
    _a, _b = (STORE[l] for l in _labs)
    _same_cols = list(_a["proteins"]) == list(_b["proteins"])
    _ka = _a["df"].set_index([_a["id"], _a["time"]]).sort_index()
    _kb = _b["df"].set_index([_b["id"], _b["time"]]).sort_index()
    _same_keys = _ka.index.equals(_kb.index)
    _same_vals = bool(_same_cols and _same_keys and
                      _ka[_a["proteins"]].equals(_kb[_b["proteins"]]))
    check("C10", "A", "protein blocks identical in both files (cell by cell)",
          _same_vals, f"same columns={_same_cols}, same patient-visit keys={_same_keys}")
    if not _same_vals:
        print("  The two files differ in the protein block, so imputing once is "
              "not valid.\n  Everything below uses the first file "
              f"({_labs[0]}); report this difference.")
else:
    check("C10", "A", "protein blocks identical in both files", False,
          "only one dataset loaded")

# ---- (b) visit structure -------------------------------------------------
_df, _id, _t = REF["df"], REF["id"], REF["time"]
POINTS = sorted(pd.unique(_df[_t].dropna()))
_has = (_df.assign(_x=1).pivot_table(index=_id, columns=_t, values="_x",
                                     aggfunc="size", fill_value=0)
        .reindex(columns=POINTS, fill_value=0) > 0)
_pat = _has.astype(int).astype(str).agg("".join, axis=1)
_pt_label = "".join(str(int(p)) if float(p).is_integer() else str(p) for p in POINTS)
_patterns = (_pat.value_counts().rename_axis(f"pattern {_pt_label}")
             .reset_index(name="patients"))
_patterns["visits"] = _patterns.iloc[:, 0].str.count("1")
show_table(_patterns, "Visit patterns (1 = sample present at that sampling point)",
           index=False, csv="s1_visit_patterns.csv")

_n_complete = int((_has.sum(axis=1) == len(POINTS)).sum())
_n_incomplete = int(len(_has) - _n_complete)
_absent = (~_has).sum(axis=0)                      # lost samples per point
_absent_fu = _absent[_absent.index != POINTS[0]]   # follow-up points only
if _absent_fu.sum() > 0 and len(_absent_fu) > 1:
    _chi = _stats.chisquare(_absent_fu.to_numpy())
    _chi_stat, _chi_p = float(_chi.statistic), float(_chi.pvalue)
else:
    _chi_stat, _chi_p = float("nan"), float("nan")
print(f"  {_n_complete} patients have all {len(POINTS)} points, "
      f"{_n_incomplete} have fewer")
print(f"  lost samples by follow-up point: "
      + ", ".join(f"point {p}: {int(n)}" for p, n in _absent_fu.items()))
print(f"  uniform-loss chi-square = {_chi_stat:.2f}, p = {_chi_p:.3f} "
      "(a large p means no point loses more samples than another)")
check("C11", "A", "every patient keeps the baseline sampling point",
      bool(_has[POINTS[0]].all()))

# protein missingness by sampling point
_P = _df[REF["proteins"]]
_mbp = (_P.isna().groupby(_df[_t]).mean().mean(axis=1) * 100).round(2)
_mbp_tab = _mbp.rename_axis(_t).reset_index(name="missing %")
show_table(_mbp_tab, "Protein missingness by sampling point", index=False,
           precision=2, csv="s1_missing_by_point.csv")
_spread = float(_mbp.max() - _mbp.min())
print(f"  spread across points: {_spread:.2f} percentage points")

num("SamplingPoints", ", ".join(_pt_label))
num("NSamplingPoints", len(POINTS))
num("NComplete", _n_complete)
num("NIncomplete", _n_incomplete)
num("VisitUniformP", round(_chi_p, 2), ".2f")
num("MissSpreadPP", round(_spread, 2), ".2f")
print("CELL 7 OK")

### CELL 8   STAGE 1.4   ENDPOINT INTEGRITY, AND THE CHECK LIST SO FAR

In [ ]:
# ======================================================================
#  CELL 8   STAGE 1.4   ENDPOINT INTEGRITY, AND THE CHECK LIST SO FAR
#  RFS (relapse-free survival) counts any recurrence, local or distant,
#  so every MFS (metastasis) event must also be an RFS event, no later
#  than the metastasis. These checks form gate B, which decides whether
#  the survival stage (CELL 24) may run. With SURVIVAL_MODE = "off" (the
#  default since the revision of 19 October 2026) they are not run.
#
#  The survival time in these files is stored per visit and shrinks from
#  visit to visit (time from that sample to the event). The analysis uses
#  the first visit, i.e. time from baseline.
# ======================================================================
def _per_patient(st):
    """One row per patient (first visit), plus the patients whose event
    flag changes between visits or whose time increases between visits."""
    d = st["df"].sort_values([st["id"], st["time"]])
    g = d.groupby(st["id"])
    out = g[[st["surv"], st["event"]]].first()
    ev_varies = g[st["event"]].nunique()[lambda s: s > 1].index.tolist()
    t_rises = g[st["surv"]].apply(
        lambda s: bool((s.dropna().diff().dropna() > 0).any()))
    t_rises = t_rises[t_rises].index.tolist()
    gaps = out.index[out.isna().any(axis=1)].tolist()
    return out, ev_varies, t_rises, gaps


GATE_B_OK = False
ENDPOINTS = None
if SURVIVAL_MODE == "off":
    print("  survival is not part of this analysis (SURVIVAL_MODE = 'off' in CELL 2): "
          "the endpoint checks (gate B) are not run")
elif "RFS" in STORE and "MFS" in STORE and all(
        STORE[l]["surv"] and STORE[l]["event"] for l in ("RFS", "MFS")):
    _R, _evR, _upR, _gR = _per_patient(STORE["RFS"])
    _M, _evM, _upM, _gM = _per_patient(STORE["MFS"])
    ENDPOINTS = pd.concat([_R.set_axis(["t_RFS", "e_RFS"], axis=1),
                           _M.set_axis(["t_MFS", "e_MFS"], axis=1)], axis=1)
    ENDPOINTS.index.name = STORE["RFS"]["id"]
    ENDPOINTS.to_csv(DIRS["results"] / "s1_endpoint_pairs.csv")

    def _ids(x):
        return f"{x[:12]}{' ...' if len(x) > 12 else ''}"

    _e1 = [f"{lab}: {what} for patients {_ids(ids)}"
           for lab, what, ids in (("RFS", "event flag changes", _evR),
                                  ("MFS", "event flag changes", _evM),
                                  ("RFS", "time increases between visits", _upR),
                                  ("MFS", "time increases between visits", _upM),
                                  ("RFS", "time or event missing", _gR),
                                  ("MFS", "time or event missing", _gM)) if ids]
    _ev_ok = all(set(pd.unique(ENDPOINTS[c].dropna())) <= {0, 1}
                 for c in ("e_RFS", "e_MFS"))
    _contra = ENDPOINTS[(ENDPOINTS.e_MFS == 1) & (ENDPOINTS.e_RFS == 0)]
    _order = ENDPOINTS[(ENDPOINTS.e_MFS == 1) & (ENDPOINTS.e_RFS == 1) &
                       (ENDPOINTS.t_RFS > ENDPOINTS.t_MFS)]
    _nonpos = ENDPOINTS[(ENDPOINTS.t_RFS <= 0) | (ENDPOINTS.t_MFS <= 0)]

    _r = [check("E1", "B", "event fixed per patient, time never increasing across visits",
                not _e1, "; ".join(_e1)),
          check("E2", "B", "events coded 0/1", _ev_ok),
          check("E3", "B", "every MFS event is also an RFS event",
                len(_contra) == 0,
                f"{len(_contra)} patients have a metastasis but no recurrence"),
          check("E4", "B", "recurrence no later than metastasis (both events)",
                len(_order) == 0, f"{len(_order)} patients with t_RFS > t_MFS"),
          check("E5", "B", "all survival times positive", len(_nonpos) == 0,
                f"{len(_nonpos)} patients with a time of zero or less")]
    GATE_B_OK = all(_r)

    _xt = pd.crosstab(ENDPOINTS.e_RFS.rename("RFS event"),
                      ENDPOINTS.e_MFS.rename("MFS event"))
    show_table(_xt, "Events per patient: RFS (rows) against MFS (columns)",
               csv="s1_event_crosstab.csv")
    if len(_nonpos):
        show_table(_nonpos, "Patients with a non-positive survival time",
                   csv="s1_nonpositive_times.csv")
    if len(_order):
        show_table(_order, "Patients whose recurrence is dated after their metastasis",
                   csv="s1_endpoint_order.csv")

    num("RFSEvents", int(ENDPOINTS.e_RFS.sum()))
    num("MFSEvents", int(ENDPOINTS.e_MFS.sum()))
    num("ContradictoryPatients", len(_contra))
    num("OrderViolations", len(_order))
    num("NonPositiveTimes", len(_nonpos))
else:
    check("E0", "B", "both endpoints with time and event available", False)

num("RFSSha", STORE["RFS"]["sha256"][:16])
num("MFSSha", STORE["MFS"]["sha256"][:16])

# ---- every check so far, with its gate ----------------------------------
_ck = pd.DataFrame(CHECKS)
GATE_A_OK = bool(_ck[_ck.gate == "A"].passed.all())
show_table(_ck[["id", "gate", "label", "passed"]], "Every check so far",
           index=False, csv="s1_checks.csv")
print(f"\n  Gate A (protein block, needed for Stages 2-4): "
      f"{'PASSED' if GATE_A_OK else 'FAILED'}")
if SURVIVAL_MODE == "off":
    print("  Gate B (endpoints): not run, survival is off")
else:
    print(f"  Gate B (endpoints, needed for Stage 5 survival): "
          f"{'PASSED' if GATE_B_OK else 'FAILED - the RFS/MFS columns are not used as they are; CELL 24 uses the reconciled endpoint (SURVIVAL_MODE)'}")
if not GATE_A_OK:
    raise RuntimeError("Gate A failed: the protein block did not pass its checks "
                       "(see FAIL above). Stages 2-4 would not be valid.")
print("CELL 8 OK")

### CELL 9   STAGE 2   TRANSFORM AND AMPUTATION   (+ Figure 2)

In [ ]:
# ======================================================================
#  CELL 9   STAGE 2   TRANSFORM AND AMPUTATION   (+ Figure 2)
#  Ground truth: mask 20% of the OBSERVED cells under MCAR, MAR and MNAR,
#  keep their true values aside, and score every imputer on them.
# ======================================================================
# ---- the analysis matrix: rows ordered by patient, then sampling point ----
_d = REF["df"].sort_values([REF["id"], REF["time"]]).reset_index(drop=True)
_pm_all = REF["pmiss"]
PROTEINS = [p for p in REF["proteins"] if _pm_all[p] <= MAX_PCT_MISSING]
if SELECT_BY == "cv":
    _raw = _d[PROTEINS]
    PROTEINS = list((_raw.std() / _raw.mean().abs()).sort_values(ascending=False)
                    .index[:N_PROTEINS_SEL])
elif SELECT_BY == "observed":
    PROTEINS = list(_pm_all[PROTEINS].sort_values().index[:N_PROTEINS_SEL])
PATIENT = _d[REF["id"]].to_numpy()
TIMEPT = _d[REF["time"]].to_numpy()
CLIN = _d[[c for c, _, _, why in REF["clinical"] if why != "survival"]]

_raw = _d[PROTEINS].to_numpy(dtype=float)
if LOG_TRANSFORM:
    _bad = int(np.sum(_raw <= -1))
    if _bad:
        print(f"  {_bad} values <= -1 cannot be log-transformed; treated as missing")
    with np.errstate(invalid="ignore", divide="ignore"):
        X = np.where(_raw > -1, np.log2(_raw + 1), np.nan)
else:
    X = _raw.copy()
OBS = np.isfinite(X)                                   # observed before masking
n, p = X.shape
print(f"  analysis matrix: {n} rows x {p} proteins, {int(OBS.sum())} observed cells")

# column-standardised values (observed cells only)
_mu = np.nanmean(X, axis=0)
_sd = np.nanstd(X, axis=0)
_sd[~np.isfinite(_sd) | (_sd < 1e-12)] = 1.0
Z = (X - _mu) / _sd


# The masking itself lives in the mipy package (mipy.masking.Amputer). It
# reproduces MIPY v1 exactly, so the masks, and the saved results that
# depend on them, are identical; v2 adds a hard detection-limit design (LOD).
AMP = mipy.masking.Amputer(X, time=TIMEPT, clinical=CLIN, rate=AMPUTE_RATE,
                           strength=AMP_STRENGTH, min_obs=MIN_OBS_PER_PROTEIN)


def ampute(mech, seed, strength=None):
    return AMP.ampute(mech, seed, strength)


AMPUTED, _rows = {}, []
for _j, mech in enumerate(MECHANISMS):
    AMPUTED[mech] = ampute(mech, SEED + 101 * (_j + 1))
    _rows.append({"mechanism": mech, **AMP.summary(AMPUTED[mech])})
AMP_TABLE = pd.DataFrame(_rows)
show_table(AMP_TABLE, "Amputation summary", index=False,
           csv="s2_amputation.csv")

_z = dict(zip(AMP_TABLE.mechanism, AMP_TABLE["mean z of masked cells"]))
check("A1", "A", "MCAR masks values at random (|mean z| < 0.1)",
      abs(_z.get("MCAR", 0)) < 0.1, f"mean z = {_z.get('MCAR')}")
check("A2", "A", "MNAR removes low values (mean z < -0.3)",
      _z.get("MNAR", 0) < -0.3, f"mean z = {_z.get('MNAR')}")
check("A3", "A", f"every protein keeps at least {MIN_OBS_PER_PROTEIN} observed cells",
      all(int((~np.isnan(AMPUTED[m]["X"][:, k])).sum()) >=
          min(MIN_OBS_PER_PROTEIN, int(OBS[:, k].sum()))
          for m in AMPUTED for k in range(p)))

# ---- Figure 2: standardised value of the masked cells -------------------
fig, axes = plt.subplots(1, len(MECHANISMS), figsize=(9.5, 2.9), sharey=True)
_bins = np.linspace(-4, 4, 41)
for ax, mech, col in zip(np.atleast_1d(axes), MECHANISMS, MECH_COLOURS):
    zm = Z[AMPUTED[mech]["mask"]]
    ax.hist(np.clip(Z[OBS], -4, 4), bins=_bins, density=True,
            color=OKABE["grey"], alpha=0.25, label="all observed")
    ax.hist(np.clip(zm, -4, 4), bins=_bins, density=True, color=col,
            alpha=0.8, label="masked")
    ax.axvline(zm.mean(), color=OKABE["red"], lw=1.2)
    ax.set_title(f"{mech}   mean z = {zm.mean():+.2f}", loc="left", fontsize=9)
    ax.set_xlabel("standardised log abundance (z)")
axes[0].set_ylabel("density")
axes[0].legend(fontsize=7)
show_fig(fig, "fig02_mechanisms")

num("LogTransform", r"$\log_2(x+1)$", raw=True)
num("NProteinsAnalysed", p)
num("NCellsObserved", f"{int(OBS.sum()):,}".replace(",", "{,}"), raw=True)
num("AmputeRatePct", round(100 * AMPUTE_RATE))
num("AmpStrength", AMP_STRENGTH, ".1f")
num("MinObsPerProtein", MIN_OBS_PER_PROTEIN)
for mech in MECHANISMS:
    num(f"MaskedZ{mech}", _z[mech], ".2f")
    num(f"MaskedPct{mech}", float(AMP_TABLE.set_index("mechanism")
                                  .loc[mech, "% of observed"]), ".1f")
print("CELL 9 OK")

### CELL 10   STAGE 3.1   METHODS, PREDICTOR SETS AND TABLE 1

In [ ]:
# ======================================================================
#  CELL 10   STAGE 3.1   METHODS, PREDICTOR SETS AND TABLE 1
#  All imputers now come from the mipy package. The thirteen methods of
#  v1 are reproduced exactly (verified to the last digit), so results
#  saved by v1 are reused. New in v2:
#    MICE-Long      chained equations that also use the patient's own
#                   previous visit, next visit and mean (cluster bootstrap,
#                   proper multiple imputation)
#    MICE-LongMNAR  MICE-Long with selection-model draws for left-censoring
#                   (sensitivity parameter gamma = MNAR_GAMMA in CELL 2)
#  The multiple-imputation versions of softImpute, missForest and
#  miceforest run in CELL 16.
# ======================================================================
PKG = mipy.METHODS
METHOD_REV = {k: v.rev for k, v in PKG.items() if v.rev}


def make_ctx(schema=None, rows=None, **kw):
    """Settings for one imputation run, from CELL 2 and the analysis matrix."""
    pat = PATIENT if rows is None or len(PATIENT) == rows else np.arange(rows)
    tim = TIMEPT if rows is None or len(TIMEPT) == rows else None
    return mipy.ImputeContext(seed=SEED, patient=pat, time=tim, schema=schema,
                              mice_iter=MICE_ITER, m=N_IMPUTATIONS, knn_k=KNN_K,
                              max_predictors=MAX_PREDICTORS, min_cor=MIN_COR,
                              gain_epochs=GAIN_EPOCHS, hyper_inner_iter=HYPER_INNER_ITER,
                              gamma=MNAR_GAMMA, **kw)


def _wrap(name):
    """v1-style call fn(X, schema=...) used by the scaling stage (CELL 18),
    which swaps PATIENT for simulated patients."""
    def f(Xm, schema=None, **kw):
        pat = PATIENT if len(PATIENT) == len(Xm) else np.arange(len(Xm))
        ctx = make_ctx(schema, rows=len(Xm)).with_(patient=pat)
        return PKG[name].fn(Xm, ctx)
    return f


# Evidence behind each method AS IMPLEMENTED (Table 2 of the manuscript):
#   "yes"      a peer-reviewed publication describes the method as implemented
#   "adapted"  a published method changed in engine, stopping rule or unit of
#              application: MICE-Bayes (Bayesian ridge instead of mice's least
#              squares), missForest (extremely randomised trees, five fixed
#              sweeps instead of the stopping rule), MinProb and QRILC (per
#              protein instead of imputeLCMD's per sample)
#   "no"       no peer-reviewed description (this includes the methods new here)
ADAPTED = {"MICE-Bayes", "missForest", "MinProb", "QRILC"}


def evidence(name):
    if name in ADAPTED:
        return "adapted"
    if PKG[name].new:
        return "no"
    return "yes" if PKG[name].peer_reviewed else "no"


#   v1 layout: (function, family, peer, package, function called, bib key, needs)
METHODS = {k: (_wrap(k), v.family, v.peer_reviewed, v.package, v.function, v.cite, v.needs)
           for k, v in PKG.items() if v.main}
MULTIPLE = {k for k, v in PKG.items() if v.main and v.multiple}
NEW_METHODS = {k for k, v in PKG.items() if v.new}
QUICK_SET = ["Mean", "Median", "LOCF", "KNN", "softImpute", "MICE-Bayes", "MinProb",
             "QRILC", "MICE-Long", "MICE-LongMNAR"]

RUN_METHODS = QUICK_SET if QUICK else list(METHODS)
_rows = []
for name in METHODS:
    fn, fam, peer, pkg, func, cite, needs = METHODS[name]
    avail = needs is None or ENV.get(needs) is not None
    status = ("will run" if (avail and name in RUN_METHODS)
              else "not in QUICK set" if avail
              else f"skipped: {needs} failed Part 1's test" if needs in _UNUSABLE
              else f"skipped: {needs} not installed")
    _rows.append({"method": name, "family": fam, "package": pkg,
                  "function": func, "peer-reviewed": evidence(name),
                  "multiple imputation": "yes" if name in MULTIPLE else "no",
                  "new in MIPY": "yes" if name in NEW_METHODS else "no",
                  "status": status})
TABLE1 = pd.DataFrame(_rows)
show_table(TABLE1, "Table 1. Imputation methods", index=False,
           csv="table1_methods.csv")
RUN_METHODS = [m for m in RUN_METHODS
               if METHODS[m][6] is None or ENV.get(METHODS[m][6]) is not None]
MI_RUN = [m for m in MI_METHODS
          if PKG[m].needs is None or ENV.get(PKG[m].needs) is not None]

# one predictor schema per mechanism: the response indicators differ
SCHEMAS = {mech: mipy.build_variable_schema(AMPUTED[mech]["X"], MAX_PREDICTORS, MIN_COR)
           for mech in MECHANISMS}
SCHEMA = SCHEMAS[MECHANISMS[0]]
for mech in MECHANISMS:
    _sizes = [len(v) for v in SCHEMAS[mech].values()]
    print(f"  {mech:<5} predictor sets: median {int(np.median(_sizes))}, "
          f"min {min(_sizes)}, max {max(_sizes)} predictors per protein")
num("MaxPredictors", MAX_PREDICTORS)
num("MinCor", MIN_COR, ".2f")
num("MiceIter", MICE_ITER)
num("MImputations", N_IMPUTATIONS)
num("KnnK", KNN_K)
num("MnarGamma", MNAR_GAMMA, ".1f")
num("NNewMethods", len([m for m in RUN_METHODS if m in NEW_METHODS]))
print(f"CELL 10 OK   {len(RUN_METHODS)} methods will run "
      f"({len([m for m in RUN_METHODS if m in NEW_METHODS])} new in v2); "
      f"{len(MI_RUN)} multiple-imputation methods for CELL 16")


def build_variable_schema(Xa):           # used by CELL 18 (v1 name)
    return mipy.build_variable_schema(Xa, MAX_PREDICTORS, MIN_COR)

### CELL 11   STAGE 3.2   RUN THE IMPUTATIONS   (the long cell)

In [ ]:
# ======================================================================
#  CELL 11   STAGE 3.2   RUN THE IMPUTATIONS   (the long cell)
#  Every method x mechanism, plus every method on the real (unmasked)
#  data for the survival stage and the real-missingness check. Runs in
#  N_WORKERS processes; each finished run is saved at once and reused on
#  a re-run (including runs saved by MIPY v1).
# ======================================================================
_settings = (SEED, MICE_ITER, N_IMPUTATIONS, MAX_PREDICTORS, MIN_COR, KNN_K,
             SOFT_RANK, GAIN_EPOCHS, HYPER_INNER_ITER, LOG_TRANSFORM,
             SELECT_BY, N_PROTEINS_SEL, AMPUTE_RATE, AMP_STRENGTH,
             MIN_OBS_PER_PROTEIN)
STORE_CK = mipy.runner.CheckpointStore(DIRS["ckpt"], fresh=FRESH)
SCHEMA_REAL = mipy.build_variable_schema(X, MAX_PREDICTORS, MIN_COR)


def _settings_for(name):
    # v1 fingerprints are kept for v1 methods; new methods add gamma
    return _settings + ((MNAR_GAMMA,) if name == "MICE-LongMNAR" else ())


_tasks = []
for mech in list(MECHANISMS) + ["REAL"]:
    Xa = X if mech == "REAL" else AMPUTED[mech]["X"]
    sch = SCHEMA_REAL if mech == "REAL" else SCHEMAS[mech]
    for name in RUN_METHODS:
        key = f"{mech}__{name}"
        fp = mipy.runner.fingerprint(Xa, name + "|" + METHOD_REV.get(name, ""), _settings_for(name))
        _tasks.append({"key": key, "name": name, "X": Xa, "ctx": make_ctx(sch),
                       "ckpt": f"imp_{key}_{fp}", "slow": name in SLOW_METHODS})
_tasks.sort(key=lambda t: not t["slow"])            # longest jobs first
_total = len(_tasks)
_count = [0]
_log = []


def _progress(t, rec, loaded):
    _count[0] += 1
    mech, name = t["key"].split("__")
    status = "loaded" if loaded else ("ok" if rec["ok"] else "FAILED " + rec["error"][:60])
    print(f"  [{_count[0]:3d}/{_total}] {mech:<5} {name:<14} {status:<8}"
          + ("" if loaded else f" {rec['seconds']:8.1f}s"), flush=True)
    _log.append((mech, name, "loaded from disk" if loaded else
                 ("ok" if rec["ok"] else rec["error"][:60]), rec["seconds"]))


print(f"  {_total} runs: {len(RUN_METHODS)} methods x {len(MECHANISMS)} mechanisms "
      f"+ real data, {N_WORKERS} worker process(es)")
_t_all = time.time()
_res = mipy.runner.run_tasks(_tasks, STORE_CK, n_workers=N_WORKERS, on_done=_progress)
RESULTS = {k: v for k, v in _res.items() if not k.startswith("REAL__")}
RESULTS_REAL = {k.split("__", 1)[1]: v for k, v in _res.items() if k.startswith("REAL__")}

_runlog = pd.DataFrame(_log, columns=["mechanism", "method", "status", "seconds"])
show_table(_runlog.sort_values(["mechanism", "method"]), "Imputation runs", precision=1,
           index=False, csv="s3_runs.csv")
_ok = sum(r["ok"] for r in RESULTS.values())
num("NMethods", len(RUN_METHODS))
num("NRuns", len(RESULTS))
num("NRunsOK", _ok)
num("TotalImputeHours", round(sum(r["seconds"] for r in RESULTS.values()) / 3600, 2), ".2f")
_skipped = sorted(set(METHODS) - set(RUN_METHODS))
num("SkippedMethods", ", ".join(_skipped) if _skipped else "none")
check("R1", "A", "every imputation run finished", _ok == len(RESULTS),
      "" if _ok == len(RESULTS) else
      "failed: " + ", ".join(k for k, r in RESULTS.items() if not r["ok"]))
_okr = sum(r["ok"] for r in RESULTS_REAL.values())
check("R1b", "A", "every method completed the real (unmasked) data", _okr == len(RESULTS_REAL),
      "" if _okr == len(RESULTS_REAL) else
      "failed: " + ", ".join(k for k, r in RESULTS_REAL.items() if not r["ok"]))
print(f"CELL 11 OK   {_ok} of {len(RESULTS)} masked runs and {_okr} of {len(RESULTS_REAL)} "
      f"real-data runs succeeded ({(time.time() - _t_all) / 60:.1f} min this time)")

### CELL 12   STAGE 4.1   PER-PROTEIN EVALUATION   (Tables 2 and 3)

In [ ]:
# ======================================================================
#  CELL 12   STAGE 4.1   PER-PROTEIN EVALUATION   (Tables 2 and 3)
#  Every accuracy number is computed within a protein on its masked
#  cells, then summarised across proteins by the median.
#  NRMSE = RMSE / SD of that protein's true values: 1.00 = no better than
#  the protein's own mean.
#
#  Fix in this version: the within-patient structure ratio (SD ratio) now
#  compares the completed matrix with the TRUE values on the same cells,
#  pooled over patient-protein groups that contain a masked cell. Before,
#  it divided by the spread of the already-masked matrix and averaged
#  ratios with near-zero denominators, which gave 2.6-9 for almost every
#  method. Now 1.00 = within-patient variation preserved, below 1 =
#  flattened, above 1 = inflated.
# ======================================================================
def evaluate_per_protein(truth, imputed, mask):
    rows = []
    for k in range(truth.shape[1]):
        sel = mask[:, k]
        if sel.sum() < 3:
            continue
        t, q = truth[sel, k], imputed[sel, k]
        g = np.isfinite(t) & np.isfinite(q)
        if g.sum() < 3:
            continue
        t, q = t[g], q[g]
        sd = np.std(t)
        rmse = float(np.sqrt(np.mean((t - q) ** 2)))
        r = (float(np.corrcoef(t, q)[0, 1])
             if np.std(t) > 1e-12 and np.std(q) > 1e-12 else np.nan)
        rows.append({"protein": PROTEINS[k], "n_masked": int(g.sum()),
                     "rmse": rmse, "nrmse": rmse / sd if sd > 1e-12 else np.nan,
                     "mae": float(np.mean(np.abs(t - q))), "r": r,
                     "bias": float(np.mean(q - t)), "sd_truth": float(sd),
                     "sd_imputed": float(np.std(q))})
    return pd.DataFrame(rows)


def structure_ratio(X_true, X_imp, mask):
    """Pooled within-patient SD of the completed matrix divided by that of
    the true values, on the cells whose true value is known, over the
    patient-protein groups with at least 2 such cells and at least one
    masked cell (elsewhere the two agree by construction)."""
    known = np.isfinite(X_true)
    num_ = den = 0.0
    for s in np.unique(PATIENT):
        rws = PATIENT == s
        T = np.where(known[rws], X_true[rws], np.nan)
        Q = np.where(known[rws], X_imp[rws], np.nan)
        use = (known[rws].sum(0) >= 2) & mask[rws].any(0)
        if not use.any():
            continue
        T, Q = T[:, use], Q[:, use]
        num_ += np.nansum((Q - np.nanmean(Q, 0)) ** 2)
        den += np.nansum((T - np.nanmean(T, 0)) ** 2)
    return float(np.sqrt(num_ / den)) if den > 0 else np.nan


PER_PROTEIN, _summary = {}, []
for mech in MECHANISMS:
    amp = AMPUTED[mech]
    for name in RUN_METHODS:
        rec = RESULTS.get(f"{mech}__{name}")
        if rec is None or not rec["ok"]:
            continue
        pp = evaluate_per_protein(amp["truth"], rec["imputed"], amp["mask"])
        PER_PROTEIN[(mech, name)] = pp
        pp.to_csv(DIRS["results"] / f"s4_pp_{mech}_{name}.csv", index=False)
        _summary.append({
            "mechanism": mech, "method": name, "family": METHODS[name][1],
            "peer_reviewed": evidence(name) == "yes", "evidence": evidence(name),
            "NRMSE_median": float(pp["nrmse"].median()),
            "NRMSE_IQR": float(pp["nrmse"].quantile(.75) - pp["nrmse"].quantile(.25)),
            "r_median": float(pp["r"].median()),
            "MAE_median": float(pp["mae"].median()),
            "bias_median": float(pp["bias"].median()),
            "structure_ratio": structure_ratio(X, rec["imputed"], amp["mask"]),
            "pct_nrmse_below_1": float(100 * (pp["nrmse"] < 1.0).mean()),
            "seconds": rec["seconds"], "m": rec["m"]})

SUMMARY = (pd.DataFrame(_summary).sort_values(["mechanism", "NRMSE_median"])
           .reset_index(drop=True))
SUMMARY.to_csv(DIRS["results"] / "s4_summary.csv", index=False)

PIVOT = (SUMMARY.pivot(index="method", columns="mechanism", values="NRMSE_median")
         .reindex(columns=[m for m in MECHANISMS if m in SUMMARY.mechanism.unique()]))
PIVOT = PIVOT.sort_values(PIVOT.columns[0])
_t2 = PIVOT.copy()
_t2["family"] = [METHODS[m][1] for m in _t2.index]
_t2["peer-reviewed"] = [evidence(m) for m in _t2.index]
show_table(_t2, "Table 2. Median per-protein NRMSE (1.00 = protein's own mean)",
           gradient=list(PIVOT.columns), csv="table2_nrmse.csv")

_t3 = SUMMARY[["mechanism", "method", "NRMSE_median", "r_median", "MAE_median",
               "bias_median", "structure_ratio", "pct_nrmse_below_1", "seconds"]]
show_table(_t3, "Table 3. Full evaluation (within-protein, median across proteins; "
                "structure ratio 1.00 = within-patient variation preserved)",
           gradient=["NRMSE_median"], index=False, csv="table3_full.csv")

for mech in MECHANISMS:
    sub = SUMMARY[SUMMARY.mechanism == mech]
    if sub.empty:
        continue
    num(f"Best{mech}", sub.iloc[0]["method"])
    num(f"Best{mech}NRMSE", round(sub.iloc[0]["NRMSE_median"], 3), ".3f")
    num(f"Best{mech}R", round(sub.iloc[0]["r_median"], 3), ".3f")
    peer = sub[sub.peer_reviewed]
    if not peer.empty:
        num(f"BestPeer{mech}", peer.iloc[0]["method"])
        num(f"BestPeer{mech}NRMSE", round(peer.iloc[0]["NRMSE_median"], 3), ".3f")
    old = sub[~sub.method.isin(NEW_METHODS)]          # best among the v1 methods
    if not old.empty:
        num(f"BestOld{mech}", old.iloc[0]["method"])
        num(f"BestOld{mech}NRMSE", round(old.iloc[0]["NRMSE_median"], 3), ".3f")
    for nm in NEW_METHODS:
        row = sub[sub.method == nm]
        if not row.empty:
            tag = nm.replace("-", "")
            num(f"{tag}{mech}NRMSE", round(float(row.iloc[0]["NRMSE_median"]), 3), ".3f")
            num(f"{tag}{mech}Rank", int(sub.method.tolist().index(nm)) + 1)
    _phi = sub.set_index("method")["structure_ratio"]
    num(f"PhiMin{mech}", round(float(_phi.min()), 2), ".2f")
    num(f"PhiMin{mech}Method", _phi.idxmin())
    num(f"PhiMax{mech}", round(float(_phi.max()), 2), ".2f")
    num(f"PhiMax{mech}Method", _phi.idxmax())
_ev = [evidence(m) for m in RUN_METHODS if m not in NEW_METHODS]
num("NPeerReviewed", _ev.count("yes"))
num("NAdapted", _ev.count("adapted"))
num("NNotPeerReviewed", _ev.count("no"))
num("NExistingMethods", len([m for m in RUN_METHODS if m not in NEW_METHODS]))
num("NSingleImputers", sum(m not in MULTIPLE for m in RUN_METHODS))

_base = SUMMARY[SUMMARY.method.isin(["Mean", "Median"])]
check("D0", "A", "unconditional baselines score NRMSE near 1 under MCAR",
      bool(((_base[_base.mechanism == "MCAR"].NRMSE_median - 1).abs() < 0.1).all()))
print("CELL 12 OK")

### CELL 13   STAGE 4.2   THE SCALE ARTEFACT   (Figure 5)

In [ ]:
# ======================================================================
#  CELL 13   STAGE 4.2   THE SCALE ARTEFACT   (Figure 5)
#  The Median imputer is constant within each protein, so its
#  within-protein correlation cannot exist. Pooled across proteins it
#  still looks like a good imputer. That gap is the artefact.
# ======================================================================
_rows, _ok = [], True
for mech in MECHANISMS:
    pp = PER_PROTEIN.get((mech, "Median"))
    if pp is None:
        continue
    m = AMPUTED[mech]["mask"]
    t = AMPUTED[mech]["truth"][m]
    q = RESULTS[f"{mech}__Median"]["imputed"][m]
    g = np.isfinite(t) & np.isfinite(q)
    pooled = float(np.corrcoef(t[g], q[g])[0, 1])
    within = pp["r"].median()
    _rows.append({"mechanism": mech, "pooled r (misleading)": round(pooled, 3),
                  "within-protein r": "undefined" if np.isnan(within) else round(within, 3),
                  "proteins with undefined r": int(pp["r"].isna().sum())})
    _ok &= bool(np.isnan(within) or abs(within) < 0.15)
    num(f"MedianPooledR{mech}", round(pooled, 3), ".3f")
    num(f"MedianWithinR{mech}", "undefined" if np.isnan(within) else round(float(within), 3))
check("D1", "A", "constant imputer has no within-protein correlation", _ok)
show_table(pd.DataFrame(_rows), "The Median imputer, scored two ways",
           index=False, csv="s4_scale_artefact.csv")

# ---- Figure 5 -----------------------------------------------------------
_mech0 = MECHANISMS[0]
_best = SUMMARY[SUMMARY.mechanism == _mech0].iloc[0]["method"]
if f"{_mech0}__Median" in RESULTS and (_mech0, _best) in PER_PROTEIN:
    m = AMPUTED[_mech0]["mask"]
    t = AMPUTED[_mech0]["truth"][m]
    q = RESULTS[f"{_mech0}__Median"]["imputed"][m]
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(t), size=min(6000, len(t)), replace=False)
    r_pool = float(np.corrcoef(t, q)[0, 1])

    fig, (axA, axB) = plt.subplots(1, 2, figsize=(9.5, 3.8))
    axA.scatter(t[idx], q[idx], s=3, alpha=0.25, color=OKABE["imp"], linewidths=0)
    lo, hi = np.nanmin(t), np.nanmax(t)
    axA.plot([lo, hi], [lo, hi], color=OKABE["grey"], lw=0.8, ls="--")
    axA.set_xlabel("true value (log2)")
    axA.set_ylabel("Median imputation (log2)")
    axA.set_title(f"A   Pooled over proteins: r = {r_pool:.2f}", loc="left", fontsize=9)

    r_best = PER_PROTEIN[(_mech0, _best)]["r"].dropna()
    axB.hist(r_best, bins=np.linspace(-1, 1, 41), color=OKABE["obs"], alpha=0.85,
             label=f"{_best}: median r = {r_best.median():.2f}")
    axB.axvline(r_pool, color=OKABE["imp"], lw=1.4,
                label=f"Median imputer, pooled r = {r_pool:.2f}")
    axB.text(0.02, 0.95, "Median imputer within a protein:\nr undefined for all "
             f"{int(PER_PROTEIN[(_mech0, 'Median')]['r'].isna().sum())} proteins",
             transform=axB.transAxes, va="top", fontsize=7.5, color=OKABE["grey"])
    axB.set_xlabel("within-protein correlation r")
    axB.set_ylabel("number of proteins")
    axB.set_title(f"B   Within protein ({_mech0})", loc="left", fontsize=9)
    axB.legend(fontsize=7, loc="center left")
    show_fig(fig, "fig05_scale_artefact")
print("CELL 13 OK")

### CELL 14   SUPPLEMENTARY FIGURE S2   ACCURACY BY METHOD

In [ ]:
# ======================================================================
#  CELL 14   SUPPLEMENTARY FIGURE S2   ACCURACY BY METHOD
#  The full per-protein NRMSE distribution behind each median of Table 3.
#  The dashed line at 1.0 = no better than the protein's own mean.
# ======================================================================
_order = list(PIVOT.sort_values(PIVOT.columns[0]).index)      # best (MCAR) on top
_mechs = list(PIVOT.columns)

# (The dot plot of earlier versions, fig03_nrmse, repeated Table 3 of the
#  manuscript and was dropped in the revision of 19 October 2026.)

# ---- Supplementary Figure S2: per-protein distributions ---------------------
_cap = 4.0                                         # NRMSE above this drawn at the edge
fig, axes = plt.subplots(1, len(_mechs), figsize=(10, 0.36 * len(_order) + 1.6),
                         sharey=True)
for ax, mech, col in zip(np.atleast_1d(axes), _mechs, MECH_COLOURS):
    data = [np.clip(PER_PROTEIN[(mech, mth)]["nrmse"].dropna().to_numpy(), 0, _cap)
            if (mech, mth) in PER_PROTEIN else np.array([np.nan]) for mth in _order]
    bp = ax.boxplot(data, vert=False, widths=0.6, showfliers=False,
                    patch_artist=True, medianprops=dict(color="black", lw=1))
    for b in bp["boxes"]:
        b.set_facecolor(col)
        b.set_alpha(0.75)
    ax.axvline(1.0, ls="--", lw=0.8, color=OKABE["grey"])
    ax.set_title(mech, loc="left", fontsize=9)
    ax.set_xlim(0, _cap)
    ax.set_xlabel("per-protein NRMSE")
np.atleast_1d(axes)[0].set_yticks(np.arange(1, len(_order) + 1))
np.atleast_1d(axes)[0].set_yticklabels(_order)
np.atleast_1d(axes)[0].invert_yaxis()
show_fig(fig, "fig04_nrmse_dist")

_iqr = (SUMMARY.pivot(index="method", columns="mechanism", values="NRMSE_IQR")
        .reindex(index=_order, columns=_mechs).round(3))
show_table(_iqr, "Spread across proteins: interquartile range of NRMSE",
           csv="s4_nrmse_iqr.csv")
print("CELL 14 OK   figure saved as fig04_nrmse_dist (.pdf and .png)")

### CELL 15   FIGURE 3 OF THE MANUSCRIPT   STRUCTURE PRESERVATION

In [ ]:
# ======================================================================
#  CELL 15   FIGURE 3 OF THE MANUSCRIPT   STRUCTURE PRESERVATION
#  The within-patient structure ratio of every method under each
#  mechanism. 1 = within-patient variation preserved, below 1 = flattened,
#  above 1 = inflated; the shaded band is 0.9 to 1.1.
# ======================================================================
_mechs = [m for m in MECHANISMS if m in set(SUMMARY.mechanism)]

# ---- Figure 6: structure ratio, one row per method ----------------------------
_ord6 = list(PIVOT.index)                          # same order as Table 2
_phiw = SUMMARY.pivot(index="method", columns="mechanism", values="structure_ratio")
_xmax = max(1.2, float(SUMMARY.NRMSE_median.max()) * 1.08)
fig, ax = plt.subplots(figsize=(7.2, 0.42 * len(_ord6) + 1.4))
ax.axvspan(0.9, 1.1, color=OKABE["third"], alpha=0.10, lw=0)
ax.axvline(1.0, ls="--", lw=0.8, color=OKABE["grey"])
_y6 = np.arange(len(_ord6))
for j, (mech, col) in enumerate(zip(_mechs, MECH_COLOURS)):
    ax.scatter(_phiw.reindex(_ord6)[mech], _y6 + (j - 1) * 0.22, s=30, color=col,
               label=mech, zorder=3)
ax.set_yticks(_y6)
ax.set_yticklabels(_ord6)
ax.invert_yaxis()
ax.set_xlim(0, max(1.5, float(np.nanmax(_phiw.to_numpy())) * 1.05))
ax.set_xlabel("within-patient SD ratio  (1.0 = preserved; < 1 flattened; > 1 inflated)")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=len(_mechs), fontsize=8)
show_fig(fig, "fig06_structure")

# (Figure 7 of earlier versions, fig07_peer, left out the two new methods
#  and was dropped in the revision of 19 October 2026.)

# ---- the structure ratio, method by method ------------------------------------
_phi = (SUMMARY.pivot(index="method", columns="mechanism", values="structure_ratio")
        .reindex(index=list(PIVOT.index), columns=_mechs).round(3))
show_table(_phi, "Within-patient SD ratio (1.00 = preserved; below 1 = flattened)",
           csv="s4_structure_ratio.csv")
for mech in _mechs:
    if "LOCF" in _phi.index:
        num(f"PhiLOCF{mech}", float(_phi.loc["LOCF", mech]), ".2f")
print("CELL 15 OK   figure saved as fig06_structure (.pdf and .png)")

### CELL 16   STAGE 4.3   MULTIPLE IMPUTATION   (Table 4)

In [ ]:
# ======================================================================
#  CELL 16   STAGE 4.3   MULTIPLE IMPUTATION   (Table 4)
#  Methods that return m completions: MICE-Bayes, MICE-Long and
#  MICE-LongMNAR (native), and the MI versions of softImpute, missForest
#  and miceforest (run here). For each:
#    Rubin's rules: FMI for the mean of the imputed cells and the median
#      FMI across protein means. For the mean of the imputed cells, the
#      within-imputation variance of completion l is U_l = s_l^2 / N, where
#      s_l^2 is the variance of the N imputed values of that completion
#      (masked and originally missing cells, all proteins together), i.e.
#      the squared standard error of their mean as if the N cells were an
#      independent sample (mipy.mi.fmi_summary). For each protein mean, the
#      same with the protein's own column (all its rows, observed and imputed).
#      Ubar and B of every row are saved in results/table4_mi_full.csv;
#    calibration: the share of masked cells whose true value falls inside
#      the 95% interval formed from the m completions (about 95% if the
#      spread of the completions matches the real uncertainty; single
#      imputation has no interval at all).
# ======================================================================
_mi_extra = [m for m in MI_RUN if m not in RUN_METHODS]
_tasks = []
for mech in MECHANISMS:
    Xa = AMPUTED[mech]["X"]
    for name in _mi_extra:
        key = f"{mech}__{name}"
        fp = mipy.runner.fingerprint(Xa, name + "|" + METHOD_REV.get(name, ""), _settings)
        _tasks.append({"key": key, "name": name, "X": Xa, "ctx": make_ctx(SCHEMAS[mech]),
                       "ckpt": f"mi_{key}_{fp}", "slow": PKG[name].slow})
_tasks.sort(key=lambda t: not t["slow"])
_n = [0]


def _p16(t, rec, loaded):
    _n[0] += 1
    print(f"  [{_n[0]:2d}/{len(_tasks)}] {t['key']:<26} "
          + ("loaded" if loaded else ("ok" if rec["ok"] else "FAILED " + rec["error"][:50]))
          + ("" if loaded else f" {rec['seconds']:8.1f}s"), flush=True)


RESULTS_MI = dict(mipy.runner.run_tasks(_tasks, STORE_CK, n_workers=N_WORKERS, on_done=_p16))
for mech in MECHANISMS:
    for name in MI_RUN:
        if name in RUN_METHODS and f"{mech}__{name}" in RESULTS:
            RESULTS_MI[f"{mech}__{name}"] = RESULTS[f"{mech}__{name}"]

RUBIN = []
for mech in MECHANISMS:
    amp = AMPUTED[mech]
    for name in MI_RUN:
        rec = RESULTS_MI.get(f"{mech}__{name}")
        if rec is None or not rec["ok"] or not rec["draws"] or len(rec["draws"]) < 2:
            continue
        D = rec["draws"]
        f = mipy.fmi_summary(D, amp["X"])
        cov = mipy.predictive_coverage(amp["truth"], D, amp["mask"])
        pp = mipy.evaluate_per_protein(amp["truth"], rec["imputed"], amp["mask"])
        sr = float(np.mean([mipy.structure_ratio(X, d, amp["mask"], PATIENT) for d in D]))
        RUBIN.append({"mechanism": mech, "method": name, "m": f["m"],
                      "nRMSE (mean of draws)": float(pp["nrmse"].median()),
                      "FMI, imputed cells": f["FMI"],
                      "median FMI, protein means": f["median FMI, protein means"],
                      "coverage of 95% intervals": cov["coverage"],
                      "interval width": cov["width"],
                      "SD ratio per completion": sr,
                      "Qbar": f["Qbar"], "Ubar": f["Ubar"], "B": f["B"], "T": f["T"], "df": f["df"],
                      "seconds": rec["seconds"]})
        tag = name.replace("-", "")
        num(f"Cov{tag}{mech}", round(100 * cov["coverage"], 1), ".1f")
        num(f"FMI{tag}{mech}", round(f["FMI"], 3), ".3f")
        if name == "MICE-Bayes":                       # v1 macro names kept
            num(f"FMI{mech}", round(f["FMI"], 3), ".3f")
            num(f"FMIProt{mech}", round(f["median FMI, protein means"], 2), ".2f")

if RUBIN:
    TABLE4 = pd.DataFrame(RUBIN)
    show_table(TABLE4[["mechanism", "method", "m", "nRMSE (mean of draws)", "FMI, imputed cells",
                       "median FMI, protein means", "coverage of 95% intervals",
                       "interval width", "SD ratio per completion"]],
               "Table 4. Multiple imputation: Rubin's rules and calibration",
               index=False, precision=3, csv="table4_mi.csv")
    TABLE4.to_csv(DIRS["results"] / "table4_mi_full.csv", index=False)   # with Qbar, Ubar, B, T
    check("M1", "A", "multiple imputations differ (between-imputation variance > 0)",
          bool((TABLE4.B > 0).all()))
    _mc = TABLE4[TABLE4.mechanism == "MCAR"]
    check("M2", "A", "under MCAR, proper MI intervals cover 85-99% of masked cells",
          bool(_mc["coverage of 95% intervals"].between(0.85, 0.99).all()),
          ", ".join(f"{r.method} {100 * r['coverage of 95% intervals']:.1f}%"
                    for _, r in _mc.iterrows()))
    num("NMIMethods", TABLE4.method.nunique())

    # (The coverage plot of earlier versions, fig08_mi_coverage, repeated
    #  Table 4 and was dropped in the revision of 19 October 2026.)
else:
    TABLE4 = None
    print("  no multiple-imputation results")
print(f"CELL 16 OK   {len(RUBIN)} method x mechanism results")

### CELL 17   STAGE 4.4   REPEATED MASKING   (Supplementary Table S10)

In [ ]:
# ======================================================================
#  CELL 17   STAGE 4.4   REPEATED MASKING   (Supplementary Table S10)
#  The masking is repeated with new seeds (repeat 1 = the main run) and
#  every method is re-scored. v2 repeats all methods, the four slow ones
#  included (N_REPEATS_SLOW in CELL 2), in parallel. Only the scores are
#  kept, not the matrices. Monte Carlo standard errors (MCSE) follow
#  Morris et al. (2019). Ranks and "% best" use only the maskings that
#  every method completed, so the comparison is like for like.
# ======================================================================
from functools import partial

REP_METHODS = list(RUN_METHODS) if REPEAT_SLOW else [m for m in RUN_METHODS if m not in SLOW_METHODS]
_settings_rep = _settings + ("repeat-v1",)
_rep_rows = []
for _, r in SUMMARY[SUMMARY.method.isin(REP_METHODS)].iterrows():   # repeat 1 = main run
    _rep_rows.append({"repeat": 1, "mechanism": r.mechanism, "method": r.method,
                      "NRMSE": r.NRMSE_median, "r": r.r_median, "bias": r.bias_median,
                      "structure_ratio": r.structure_ratio, "seconds": r.seconds})

_tasks = []
for rep in range(2, N_REPEATS + 1):
    for j, mech in enumerate(MECHANISMS):
        amp = ampute(mech, SEED + 1000 * rep + 101 * (j + 1))
        Xa = amp["X"]
        sch = None
        for name in REP_METHODS:
            if name in SLOW_METHODS and rep > N_REPEATS_SLOW:
                continue
            setts = _settings_rep + ((MNAR_GAMMA,) if name == "MICE-LongMNAR" else ())
            fp = mipy.runner.fingerprint(Xa, name + "|" + METHOD_REV.get(name, ""), setts)
            if sch is None:
                sch = mipy.build_variable_schema(Xa, MAX_PREDICTORS, MIN_COR)
            _tasks.append({"key": (rep, mech, name), "name": name, "X": Xa, "ctx": make_ctx(sch),
                           "ckpt": f"rep{rep}_{mech}__{name}_{fp}", "keep_draws": True,
                           "post": partial(mipy.runner.score_record, X_true=X,
                                           mask=amp["mask"], patient=PATIENT),
                           "slow": name in SLOW_METHODS})
_tasks.sort(key=lambda t: not t["slow"])
_n17 = [0]


def _p17(t, rec, loaded):
    _n17[0] += 1
    if loaded:
        return
    rep, mech, name = t["key"]
    print(f"  [{_n17[0]:4d}/{len(_tasks)}] repeat {rep:2d} {mech:<5} {name:<14}"
          + (f" ok {rec['seconds']:7.1f}s" if rec["ok"] else f" FAILED {rec.get('error', '')[:50]}"),
          flush=True)


_t_rep = time.time()
if _tasks:
    print(f"  {len(_tasks)} extra runs over {N_REPEATS - 1} new maskings "
          f"(slow methods: {N_REPEATS_SLOW - 1 if REPEAT_SLOW else 0}), {N_WORKERS} worker(s)")
_res17 = mipy.runner.run_tasks(_tasks, STORE_CK, n_workers=N_WORKERS, on_done=_p17)
print(f"  {sum(1 for t in _tasks)} repeat runs available "
      f"({(time.time() - _t_rep) / 60:.1f} min this time)")
for (rep, mech, name), rec in _res17.items():
    if rec.get("ok"):
        _rep_rows.append({"repeat": rep, "mechanism": mech, "method": name,
                          **{k: rec[k] for k in ("NRMSE", "r", "bias", "structure_ratio", "seconds")}})

REPEATS = pd.DataFrame(_rep_rows)
REPEATS.to_csv(DIRS["results"] / "s6_repeats_raw.csv", index=False)
_full = REPEATS.groupby(["repeat", "mechanism"])["method"].transform("nunique") == len(REP_METHODS)
REPEATS["rank"] = np.nan
REPEATS.loc[_full, "rank"] = (REPEATS[_full].groupby(["repeat", "mechanism"])["NRMSE"]
                              .rank(method="min"))
_g = REPEATS.groupby(["mechanism", "method"])
TABLE6 = pd.DataFrame({
    "repeats": _g["NRMSE"].size(),
    "NRMSE mean": _g["NRMSE"].mean(),
    "MCSE": _g["NRMSE"].apply(mipy.mcse_mean),
    "NRMSE SD": _g["NRMSE"].std(ddof=1),
    "NRMSE 2.5%": _g["NRMSE"].quantile(0.025),
    "NRMSE 97.5%": _g["NRMSE"].quantile(0.975),
    "SD ratio mean": _g["structure_ratio"].mean(),
    "mean rank": _g["rank"].mean(),
    "% best": _g["rank"].apply(lambda s: 100 * (s.dropna() == 1).mean() if s.notna().any() else np.nan),
}).reset_index()
TABLE6["mechanism"] = pd.Categorical(TABLE6.mechanism, list(MECHANISMS), ordered=True)
TABLE6 = TABLE6.sort_values(["mechanism", "NRMSE mean"]).reset_index(drop=True)
TABLE6["mechanism"] = TABLE6.mechanism.astype(str)
show_table(TABLE6, f"Table 6. Median NRMSE over {N_REPEATS} independent maskings",
           index=False, csv="table6_repeats.csv")

# (The plot of earlier versions, fig09_repeats, repeated Supplementary
#  Table S10 and was dropped in the revision of 19 October 2026.)

num("NRepeats", N_REPEATS)
num("NRepeatsSlow", N_REPEATS_SLOW if REPEAT_SLOW else 1)
num("NRepeatedMethods", len(REP_METHODS))
num("RepeatSlow", "yes" if REPEAT_SLOW else "no")
num("RepeatMaxSD", round(float(TABLE6["NRMSE SD"].max()), 3), ".3f")
num("RepeatMaxMCSE", round(float(TABLE6["MCSE"].max()), 4), ".4f")
for mech in MECHANISMS:
    s = TABLE6[TABLE6.mechanism == mech]
    if len(s) and s["% best"].notna().any():
        b = s.sort_values("% best", ascending=False).iloc[0]
        num(f"RepBest{mech}", b.method)
        num(f"RepBestPct{mech}", round(float(b["% best"])), ".0f")
_need = {m: (N_REPEATS_SLOW if (m in SLOW_METHODS and REPEAT_SLOW) else N_REPEATS) for m in REP_METHODS}
_have = REPEATS.groupby("method")["repeat"].nunique()
_short = [m for m in REP_METHODS if _have.get(m, 0) < _need[m]]
check("R2", "A", "every repeated run finished", not _short,
      "missing repeats: " + ", ".join(_short) if _short else
      f"{len(REP_METHODS)} methods, up to {N_REPEATS} maskings")
print(f"CELL 17 OK   {len(REPEATS)} scores")

### CELL 18   STAGE 4.5   SCALING ON SIMULATED DATA   (Table 7, Figure 10)

In [ ]:
# ======================================================================
#  CELL 18   STAGE 4.5   SCALING ON SIMULATED DATA   (Table 7, Figure 10)
#  How run time grows with the number of rows. A simulated cohort with
#  SCALING_P proteins (rank-5 signal plus noise, 4 visits per patient,
#  10% of cells missing completely at random) is imputed by every method
#  at each size in SCALING_SIZES. A method whose predicted time at the
#  next size exceeds SCALE_TIME_LIMIT seconds is not run there and is
#  reported as such. The prediction uses the growth seen between the two
#  previous sizes (linear growth is assumed for the first step).
#  Set RUN_SCALING = False in CELL 2 to skip this stage.
# ======================================================================
SCALING = None
if not RUN_SCALING:
    print("  skipped (RUN_SCALING = False in CELL 2)")
else:
    def _simulate(n_rows, p_cols, seed):
        rs = np.random.default_rng(seed)
        n_pat = int(np.ceil(n_rows / 4))
        pid = np.repeat(np.arange(n_pat), 4)[:n_rows]
        L = rs.normal(size=(n_pat, 5))[pid] + 0.3 * rs.normal(size=(n_rows, 5))
        Xs = L @ rs.normal(size=(5, p_cols)) + rs.normal(scale=0.6, size=(n_rows, p_cols)) + 8.0
        Ms = rs.random((n_rows, p_cols)) < 0.10
        return Xs, np.where(Ms, np.nan, Xs), Ms, pid

    _PATIENT_REAL = PATIENT
    _sc_rows, _hist = [], {}
    _t_sc = time.time()
    try:
        for n_rows in SCALING_SIZES:
            Xs, Xsm, Ms, pid = _simulate(n_rows, SCALING_P, SEED + 7)
            PATIENT = pid                          # LOCF groups by the simulated patients
            schema = None
            for name in RUN_METHODS:
                h = _hist.get(name, [])
                if h:
                    n0, t0_ = h[-1]
                    if len(h) >= 2:
                        b = np.log(max(h[-1][1], 1e-3) / max(h[-2][1], 1e-3)) / np.log(h[-1][0] / h[-2][0])
                        b = float(np.clip(b, 1.0, 2.5))
                    else:
                        b = 1.0
                    pred = t0_ * (n_rows / n0) ** b
                else:
                    pred = 0.0
                if h is None or pred > SCALE_TIME_LIMIT:
                    _sc_rows.append({"rows": n_rows, "method": name, "seconds": np.nan,
                                     "NRMSE": np.nan,
                                     "status": f"not run (predicted {pred / 60:.0f} min)"
                                               if h is not None else "not run (stopped at a smaller size)"})
                    _hist[name] = None                 # larger sizes are not tried either
                    print(f"  {n_rows:>9,} rows  {name:<12} not run (predicted {pred:,.0f}s)"
                          if h is not None else f"  {n_rows:>9,} rows  {name:<12} not run (stopped at a smaller size)")
                    continue
                fp = fingerprint(Xsm[:50], f"scale|{name}|{n_rows}|"
                                 + METHOD_REV.get(name, ""),
                                 _settings + (SCALING_P, "scale-v1"))
                rec = load_ckpt(f"scale_{n_rows}_{name}_{fp}")
                if rec is None:
                    if schema is None:
                        schema = build_variable_schema(Xsm)
                    t0 = time.time()
                    try:
                        out, _ = METHODS[name][0](Xsm.copy(), schema=schema)
                        out = np.asarray(out, dtype=float)
                        sec = time.time() - t0
                        nr = []
                        for k in range(SCALING_P):
                            sel = Ms[:, k]
                            sd = Xs[sel, k].std()
                            if sel.sum() >= 3 and sd > 0:
                                nr.append(np.sqrt(np.mean((out[sel, k] - Xs[sel, k]) ** 2)) / sd)
                        rec = {"seconds": sec, "NRMSE": float(np.nanmedian(nr)), "status": "ok"}
                    except Exception as e:
                        rec = {"seconds": time.time() - t0, "NRMSE": np.nan,
                               "status": f"failed: {type(e).__name__}"}
                    save_ckpt(f"scale_{n_rows}_{name}_{fp}", rec)
                print(f"  {n_rows:>9,} rows  {name:<12} {rec['status']:<8} "
                      f"{rec['seconds']:9.1f}s")
                _sc_rows.append({"rows": n_rows, "method": name, **rec})
                _hist[name] = (h or []) + [(n_rows, max(rec["seconds"], 1e-3))] \
                    if rec["status"] == "ok" else None
    finally:
        PATIENT = _PATIENT_REAL                    # the real cohort again, whatever happened

    SCALING = pd.DataFrame(_sc_rows)
    TABLE7 = (SCALING.pivot(index="method", columns="rows", values="seconds")
              .reindex(RUN_METHODS))
    TABLE7.columns = [f"{c:,} rows (s)" for c in TABLE7.columns]
    TABLE7 = TABLE7.sort_values(TABLE7.columns[0])
    show_table(TABLE7.round(2), f"Table 7. Run time on simulated data "
               f"({SCALING_P} proteins, 10% missing); blank = not run",
               csv="table7_scaling.csv")
    SCALING.to_csv(DIRS["results"] / "s7_scaling_raw.csv", index=False)

    fig, ax = plt.subplots(figsize=(7.6, 4.4))
    _cyc = [plt.cm.tab20(i) for i in (0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 1, 5, 9)]
    _mk = "osD^v<>ph*XP8"
    for i, name in enumerate(TABLE7.index):
        s = SCALING[(SCALING.method == name) & (SCALING.status == "ok")]
        if len(s):
            ax.plot(s["rows"], s["seconds"], marker=_mk[i % len(_mk)], ms=4, lw=1,
                    color=_cyc[i % len(_cyc)], label=name)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.axhline(SCALE_TIME_LIMIT, ls="--", lw=0.8, color=OKABE["grey"])
    ax.text(1.0, SCALE_TIME_LIMIT, f"time limit {SCALE_TIME_LIMIT} s ", ha="right", va="bottom",
            fontsize=7, color=OKABE["grey"], transform=ax.get_yaxis_transform())
    ax.set_xlabel("rows (samples) in the simulated matrix")
    ax.set_ylabel("run time (s)")
    ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False)
    show_fig(fig, "fig10_scaling")

    num("ScalingP", SCALING_P)
    num("ScalingMaxRows", f"{max(SCALING_SIZES):,}")
    num("ScaleTimeLimit", SCALE_TIME_LIMIT)
    _big = SCALING[(SCALING.rows == max(SCALING_SIZES)) & (SCALING.status == "ok")]
    num("NScaleMaxOK", len(_big))
    num("ScalingMinutes", round(float(SCALING["seconds"].sum()) / 60, 1), ".1f")
print(f"CELL 18 OK   scaling stage: {'done' if SCALING is not None else 'skipped'}")

### CELL 19   STAGE 4.6   HOW MUCH DOES THE MNAR ASSUMPTION MATTER?

In [ ]:
# ======================================================================
#  CELL 19   STAGE 4.6   HOW MUCH DOES THE MNAR ASSUMPTION MATTER?
#            (Table 8, Figures 11 and 12)
#  (a) The MNAR design is repeated with a weaker and a stronger slope
#      (SENS_STRENGTHS), with a probit instead of a logistic selection
#      (RUN_PROBIT; slope matched to the main design's strength) and with a
#      hard detection limit (LOD), for all methods except the four slow ones.
#  (b) MICE-LongMNAR is run over a grid of its sensitivity parameter gamma
#      (gamma = 0 is MAR) under every design, to show what assuming
#      censoring gains when it is true and costs when it is not.
#  (c) RUN_ESS: the effective sample size of MICE-LongMNAR's importance
#      resampling, (sum w)^2 / sum w^2 over its C = 50 candidates, for every
#      gamma > 0 on every design (Supplementary Section S4.1). These runs
#      repeat the gamma-grid runs with the diagnostic switched on; they
#      must reproduce the grid's accuracy exactly, and the cell says so.
# ======================================================================
SENS = {}
_mnar_seed = SEED + 101 * (MECHANISMS.index("MNAR") + 1) if "MNAR" in MECHANISMS else SEED + 303
for b in SENS_STRENGTHS:
    SENS[f"MNAR b={b:g}"] = ampute("MNAR", _mnar_seed, strength=b)
PROBIT_BETA = None
if RUN_PROBIT:
    PROBIT_BETA = (AMP.matched_probit_slope(AMP_STRENGTH) if PROBIT_SLOPE == "matched"
                   else float(PROBIT_SLOPE))
    SENS["MNAR probit"] = ampute("PROBIT", SEED + 505, strength=PROBIT_BETA)
    _ps = AMP.summary(SENS["MNAR probit"])
    _target = AMP.expected_masked_z(AMP_STRENGTH, "logit")
    print(f"  probit design: slope {PROBIT_BETA:.3f} "
          f"({'matched: expected mean z of masked cells ' + format(_target, '.3f') if PROBIT_SLOPE == 'matched' else 'set in CELL 2'}); "
          f"{_ps['cells masked']:,} cells masked ({_ps['% of observed']}% of observed), "
          f"mean z {_ps['mean z of masked cells']}")
    num("ProbitBeta", round(PROBIT_BETA, 2), ".2f")
    num("ProbitTargetZ", round(_target, 3), ".3f")
    num("ProbitMaskedZ", float(_ps["mean z of masked cells"]), ".3f")
    num("ProbitMaskedPct", float(_ps["% of observed"]), ".2f")
    num("ProbitCellsMasked", f"{int(_ps['cells masked']):,}".replace(",", "{,}"), raw=True)
if RUN_LOD:
    SENS["LOD"] = ampute("LOD", SEED + 404)
_sens_schema = {k: mipy.build_variable_schema(v["X"], MAX_PREDICTORS, MIN_COR) for k, v in SENS.items()}
_designs = {**{m: AMPUTED[m] for m in MECHANISMS}, **SENS}
MAIN_MNAR = f"MNAR b={AMP_STRENGTH:g} (main)"


def _tag(d):
    """Letters-only label for numbers.tex macro names."""
    if d.startswith("MNAR b=") and "main" not in d:
        b = float(d.split("=")[1])
        return "MNARweak" if b < AMP_STRENGTH else "MNARstrong"
    return {MAIN_MNAR: "MNARmain"}.get(d, d.replace(" ", ""))
_dschema = {**SCHEMAS, **_sens_schema}
_fast = [m for m in RUN_METHODS if m not in SLOW_METHODS]

_tasks = []
for lab, amp in SENS.items():
    for name in _fast:
        fp = mipy.runner.fingerprint(amp["X"], name + "|" + METHOD_REV.get(name, ""),
                                     _settings + (lab, MNAR_GAMMA))
        _tasks.append({"key": ("sens", lab, name, None), "name": name, "X": amp["X"],
                       "ctx": make_ctx(_dschema[lab]), "ckpt": f"sens_{lab}__{name}_{fp}".replace(" ", "").replace("=", ""),
                       "keep_draws": True,
                       "post": partial(mipy.runner.score_record, X_true=X, mask=amp["mask"], patient=PATIENT)})
for lab, amp in _designs.items():
    for g in GAMMA_GRID:
        fp = mipy.runner.fingerprint(amp["X"], "MICE-LongMNAR|" + METHOD_REV.get("MICE-LongMNAR", ""),
                                     _settings + (lab, float(g)))
        _tasks.append({"key": ("gamma", lab, "MICE-LongMNAR", float(g)), "name": "MICE-LongMNAR",
                       "X": amp["X"], "ctx": make_ctx(_dschema[lab]).with_(gamma=float(g)),
                       "ckpt": f"gam_{lab}_{g:g}_{fp}".replace(" ", "").replace("=", ""), "keep_draws": True,
                       "post": partial(mipy.runner.score_record, X_true=X, mask=amp["mask"], patient=PATIENT)})
print(f"  {len(_tasks)} runs: {len(SENS)} extra MNAR designs x {len(_fast)} methods, "
      f"and gamma grid {GAMMA_GRID} x {len(_designs)} designs")
_res19 = mipy.runner.run_tasks(_tasks, STORE_CK, n_workers=N_WORKERS)

_rows = []
for mech in MECHANISMS:                                   # main designs from Table 3
    for _, r in SUMMARY[SUMMARY.mechanism == mech].iterrows():
        if r.method in _fast:
            _rows.append({"design": MAIN_MNAR if mech == "MNAR" else mech,
                          "method": r.method, "NRMSE": r.NRMSE_median, "bias": r.bias_median})
for (kind, lab, name, g), rec in _res19.items():
    if kind == "sens" and rec.get("ok"):
        _rows.append({"design": lab, "method": name, "NRMSE": rec["NRMSE"], "bias": rec["bias"]})
SENS_TABLE = pd.DataFrame(_rows)
_order_d = [d for d in ["MCAR", "MAR"] + [f"MNAR b={b:g}" for b in sorted(SENS_STRENGTHS) if b < AMP_STRENGTH]
            + [MAIN_MNAR] + (["MNAR probit"] if RUN_PROBIT else [])      # next to the logistic
            + [f"MNAR b={b:g}" for b in sorted(SENS_STRENGTHS) if b > AMP_STRENGTH]  # design it matches
            + (["LOD"] if RUN_LOD else []) if d in set(SENS_TABLE.design)]
TABLE8 = SENS_TABLE.pivot(index="method", columns="design", values="NRMSE").reindex(columns=_order_d)
TABLE8 = TABLE8.sort_values(_order_d[-1])
show_table(TABLE8.round(3), "Table 8. Median NRMSE as the missingness becomes more strongly "
           "not at random", csv="table8_mnar_sensitivity.csv")
for d in _order_d:
    tag = _tag(d)
    best = TABLE8[d].idxmin()
    num(f"SensBest{tag}", best)
    num(f"SensBest{tag}NRMSE", round(float(TABLE8[d].min()), 3), ".3f")

GAMMA_TABLE = pd.DataFrame([{"design": MAIN_MNAR if lab == "MNAR" else lab, "gamma": g,
                             "NRMSE": rec["NRMSE"], "bias": rec["bias"]}
                            for (kind, lab, name, g), rec in _res19.items()
                            if kind == "gamma" and rec.get("ok")])
show_table(GAMMA_TABLE.pivot(index="gamma", columns="design", values="NRMSE")
           .reindex(columns=[d for d in _order_d if d in set(GAMMA_TABLE.design)]).round(3),
           "MICE-LongMNAR: median NRMSE by assumed censoring strength gamma (0 = MAR)",
           csv="s8_gamma_grid.csv")
for d, s in GAMMA_TABLE.groupby("design"):
    tag = _tag(d)
    num(f"GammaBest{tag}", float(s.loc[s.NRMSE.idxmin(), "gamma"]), ".1f")

# ---- (c) effective sample size of the importance resampling -------------------
_GWORD = {0.5: "Half", 1.0: "One", 1.5: "OneHalf", 2.0: "Two", 3.0: "Three", 4.0: "Four"}
ESS_TABLE = None
if RUN_ESS:
    _etasks = []
    for lab, amp in _designs.items():
        for g in GAMMA_GRID:
            if g <= 0:                       # gamma = 0: equal weights, ESS = C exactly
                continue
            fp = mipy.runner.fingerprint(amp["X"], "MICE-LongMNAR|" + METHOD_REV.get("MICE-LongMNAR", ""),
                                         _settings + (lab, float(g), "ess"))
            _etasks.append({"key": ("ess", lab, float(g)), "name": "MICE-LongMNAR", "X": amp["X"],
                            "ctx": make_ctx(_dschema[lab]).with_(gamma=float(g), extra={"ess": True}),
                            "ckpt": f"ess_{lab}_{g:g}_{fp}".replace(" ", "").replace("=", ""),
                            "keep_draws": True,
                            "post": partial(mipy.runner.score_record, X_true=X, mask=amp["mask"],
                                            patient=PATIENT)})
    print(f"  effective sample size: {len(_etasks)} runs (gamma > 0 x {len(_designs)} designs)")
    _res_ess = mipy.runner.run_tasks(_etasks, STORE_CK, n_workers=N_WORKERS)
    _grid = {(lab, g): rec for (kind, lab, name, g), rec in _res19.items() if kind == "gamma"}
    _erows = []
    for (kind, lab, g), rec in _res_ess.items():
        if not rec.get("ok") or "diag" not in rec:
            continue
        d = rec["diag"]
        same = _grid.get((lab, g), {}).get("NRMSE")
        _erows.append({"design": MAIN_MNAR if lab == "MNAR" else lab, "gamma": g,
                       "median ESS": d["ess_median"], "10th percentile": d["ess_p10"],
                       "1st percentile": d["ess_p01"], "minimum": d["ess_min"],
                       "% cells with ESS < 10": d["ess_below10_pct"], "cells": d["ess_cells"],
                       "NRMSE": rec["NRMSE"],
                       "|NRMSE - grid run|": abs(rec["NRMSE"] - same) if same is not None else np.nan})
    if _erows:
        ESS_TABLE = pd.DataFrame(_erows)
        _dord = [d for d in _order_d if d in set(ESS_TABLE.design)]
        ESS_TABLE["design"] = pd.Categorical(ESS_TABLE.design, _dord, ordered=True)
        ESS_TABLE = ESS_TABLE.sort_values(["gamma", "design"]).reset_index(drop=True)
        show_table(ESS_TABLE, f"MICE-LongMNAR: effective sample size of the importance resampling "
                   f"(C = 50 candidates; last sweep of each of the {N_IMPUTATIONS} completions)",
                   index=False, precision=2, csv="s4_ess_gamma.csv")
        _diff = float(np.nanmax(ESS_TABLE["|NRMSE - grid run|"]))
        print(f"  the ESS runs reproduce the gamma-grid accuracy: largest |difference| in median "
              f"NRMSE = {_diff:.2e}")
        num("EssReproMaxDiff", f"{_diff:.1e}")
        num("EssCandidates", 50)
        for g, s in ESS_TABLE.groupby("gamma"):
            w = _GWORD.get(float(g))
            if w is None:
                continue
            num(f"EssMedian{w}", round(float(s["median ESS"].median()), 1), ".1f")
            num(f"EssMedianLo{w}", round(float(s["median ESS"].min()), 1), ".1f")
            num(f"EssMedianHi{w}", round(float(s["median ESS"].max()), 1), ".1f")
            num(f"EssPtenLo{w}", round(float(s["10th percentile"].min()), 1), ".1f")
            num(f"EssBelowTenHi{w}", round(float(s["% cells with ESS < 10"].max()), 2), ".2f")

# ---- Figure 11: NRMSE against design, selected methods -------------------------
_show = [m for m in ["Mean", "LOCF", "KNN", "softImpute", "MICE-Bayes", "MICE-Long",
                     "MICE-LongMNAR", "MinProb", "QRILC"] if m in TABLE8.index]
_cyc = [plt.cm.tab10(i) for i in range(10)]
fig, ax = plt.subplots(figsize=(7.4, 4.0))
for i, m_ in enumerate(_show):
    ax.plot(range(len(_order_d)), TABLE8.loc[m_, _order_d], marker="o", ms=4, lw=1.2,
            color=_cyc[i % 10], label=m_)
ax.axhline(1.0, ls="--", lw=0.8, color=OKABE["grey"])
ax.set_xticks(range(len(_order_d)))
ax.set_xticklabels(_order_d, rotation=20, ha="right")
ax.set_ylabel("median per-protein NRMSE")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1.0))
show_fig(fig, "fig11_mnar_sensitivity")

# ---- Figure 12: gamma grid --------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.2, 3.8))
for i, d in enumerate([d for d in _order_d if d in set(GAMMA_TABLE.design)]):
    s = GAMMA_TABLE[GAMMA_TABLE.design == d].sort_values("gamma")
    ax.plot(s.gamma, s.NRMSE, marker="o", ms=4, lw=1.2, color=_cyc[i % 10], label=d)
ax.axvline(MNAR_GAMMA, ls=":", lw=0.8, color=OKABE["grey"])
ax.set_xlabel("assumed censoring strength gamma (0 = missing at random)")
ax.set_ylabel("median per-protein NRMSE")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1.0))
show_fig(fig, "fig12_gamma_sensitivity")
print(f"CELL 19 OK   {len(SENS)} extra designs; figures fig11 (Figure 4 of the manuscript) "
      f"and fig12 (Supplementary Figure S4)"
      + (f"; effective sample size for {len(ESS_TABLE)} runs" if ESS_TABLE is not None else ""))

### CELL 20   STAGE 4.7   THE REAL MISSING VALUES   (Table 9, Figure 13)

In [ ]:
# ======================================================================
#  CELL 20   STAGE 4.7   THE REAL MISSING VALUES   (Table 9, Figure 13)
#  The values that are genuinely missing in GSE65622 have no true value,
#  but two things can still be checked:
#    (a) is missingness related to abundance? Proteins with low observed
#        levels missing more often points to a detection limit (MNAR);
#    (b) where does each method put the missing values relative to the
#        observed ones (standardised mean shift; share below the lowest
#        observed value), using every method's imputation of the real data.
# ======================================================================
from scipy import stats as _stats

_pm = np.isnan(X).mean(0)
_lvl = np.nanmedian(X, 0)
_use = _pm > 0
_rho, _prho = _stats.spearmanr(_pm[_use], _lvl[_use])
print(f"  proteins with missing values: {int(_use.sum())}; Spearman correlation between "
      f"% missing and median observed level: rho = {_rho:.2f} (p = {_prho:.1e})")
num("RealMissLevelRho", round(float(_rho), 2), ".2f")
num("RealMissLevelP", f"{_prho:.1e}")

# within-patient view: is a value missing more often when the same patient's
# other visits of that protein are low?
_zrow = (X - np.nanmean(X, 0)) / np.nanstd(X, 0)
_pz = pd.DataFrame(_zrow).groupby(PATIENT).transform("mean").to_numpy()
_miss_cells = np.isnan(X) & np.isfinite(_pz)
_obs_cells = np.isfinite(X) & np.isfinite(_pz)
_dz = float(np.nanmean(_pz[_miss_cells]) - np.nanmean(_pz[_obs_cells]))
num("RealMissPatientZ", round(_dz, 2), ".2f")
print(f"  patient-level mean z of the protein: {np.nanmean(_pz[_miss_cells]):.2f} where missing, "
      f"{np.nanmean(_pz[_obs_cells]):.2f} where observed")

_rows = []
_M0 = np.isnan(X)
_cols = [k for k in range(X.shape[1]) if _M0[:, k].sum() >= 5]
_mu_o = np.nanmean(X, 0)
_sd_o = np.nanstd(X, 0)
_min_o = np.nanmin(X, 0)
for name, rec in RESULTS_REAL.items():
    if not rec.get("ok"):
        continue
    Q = rec["imputed"]
    shift = [(Q[_M0[:, k], k].mean() - _mu_o[k]) / _sd_o[k] for k in _cols]
    below = [np.mean(Q[_M0[:, k], k] < _min_o[k]) for k in _cols]
    _rows.append({"method": name, "median shift (SD units)": float(np.median(shift)),
                  "% proteins shifted down": float(100 * np.mean(np.array(shift) < 0)),
                  "median % below observed minimum": float(100 * np.median(below))})
TABLE9 = pd.DataFrame(_rows).sort_values("median shift (SD units)").reset_index(drop=True)
show_table(TABLE9, "Table 9. Where each method places the genuinely missing values "
           f"({len(_cols)} proteins with 5 or more missing samples)",
           index=False, csv="table9_real_missing.csv")
num("RealMissNProteins", len(_cols))
for _, r in TABLE9.iterrows():
    num(f"RealShift{r.method.replace('-', '')}", round(r["median shift (SD units)"], 2), ".2f")

# ---- Figure 13: observed vs imputed, the four most incomplete proteins -----------
_top = list(np.argsort(-_pm)[:4])
_meth13 = [m for m in ["softImpute", "MICE-Long", "MICE-LongMNAR", "QRILC"] if m in RESULTS_REAL
           and RESULTS_REAL[m].get("ok")]
_cyc = [OKABE["imp"], OKABE["third"], OKABE["red"], "#CC79A7"]
fig, axes = plt.subplots(1, 4, figsize=(11, 2.9), sharey=True)
for ax, k in zip(axes, _top):
    obs = np.sort(X[~_M0[:, k], k])
    ax.step(obs, np.arange(1, len(obs) + 1) / len(obs), where="post", color=OKABE["obs"],
            lw=1.6, label="observed")
    for m_, c_ in zip(_meth13, _cyc):
        q = np.sort(RESULTS_REAL[m_]["imputed"][_M0[:, k], k])
        ax.step(q, np.arange(1, len(q) + 1) / len(q), where="post", color=c_, lw=1.1, label=m_)
    ax.set_title(f"{PROTEINS[k]} ({100 * _pm[k]:.0f}% missing)", fontsize=8, loc="left")
    ax.set_xlabel(r"$\log_2$ abundance")
axes[0].set_ylabel("cumulative proportion")
axes[-1].legend(fontsize=6.5, loc="lower right")
show_fig(fig, "fig13_real_missing")
print("CELL 20 OK   figure fig13_real_missing")

### CELL 21   STAGE 4.8   AGREEMENT WITH THE ORIGINAL R PACKAGES   (Table 10)

In [ ]:
# ======================================================================
#  CELL 21   STAGE 4.8   AGREEMENT WITH THE ORIGINAL R PACKAGES   (Table 10)
#  The masked matrices are passed to R, where softImpute, missForest,
#  mice (norm.boot) and imputeLCMD (MinProb, QRILC) run on exactly the
#  same cells. Missing R packages are installed by the R script itself.
#    softImpute : same standardisation and lambda -> cell-by-cell agreement
#    imputeLCMD : the fitted parameters are deterministic -> must match
#    missForest, mice : different tree / regression engines -> accuracy
#                       and per-protein agreement
#  mice's norm.boot can stop on this data: for proteins with few observed
#  values, a bootstrap sample has fewer distinct rows than predictors, the
#  least-squares fit returns NA coefficients and mice halts. The R script
#  then reruns norm.boot with mice's own ridge option (ls.meth = "ridge"),
#  or, if that also stops, mice's "norm"; the R steps table and Table 10
#  say when this happened.
# ======================================================================
R_TABLE = None
MICE_RIDGE = MICE_NORM = False
if not RUN_R:
    print("  skipped (RUN_R = False in CELL 2)")
elif RSCRIPT_PATH is None:
    print("  skipped: R not found (install R from https://cran.r-project.org, or set RSCRIPT)")
else:
    _rdir = OUT_DIR / "r_agreement"
    _rdir.mkdir(parents=True, exist_ok=True)
    _rsrc = hashlib.sha256(Path(mipy.rbridge.r_script_path()).read_bytes()).hexdigest()[:12]
    _sig = mipy.runner.fingerprint(np.concatenate([AMPUTED[m]["X"] for m in MECHANISMS]),
                                   "r-agreement-v2", _settings + (R_MF_NTREE, R_MF_MAXITER, _rsrc))
    _st_file = _rdir / "status.csv"
    _done = (not FRESH and (_rdir / "inputs.sha").is_file()
             and (_rdir / "inputs.sha").read_text() == _sig and _st_file.is_file()
             and "done" in set(pd.read_csv(_st_file)["step"]))
    if _done:
        print("  R results found from an earlier run; not rerunning R")
    else:
        mipy.rbridge.write_inputs(_rdir, {m: AMPUTED[m] for m in MECHANISMS}, SCHEMAS, seed=SEED,
                                  m=N_IMPUTATIONS, mice_maxit=MICE_ITER, mf_ntree=R_MF_NTREE,
                                  mf_maxiter=R_MF_MAXITER)
        print(f"  running R (missForest with {R_MF_NTREE} trees is the slow part) ...", flush=True)
        _t21 = time.time()
        _status = mipy.rbridge.run_r(_rdir, RSCRIPT_PATH)
        print(f"  R finished in {(time.time() - _t21) / 60:.1f} min")
        if len(_status) and "done" in set(_status["step"]):
            (_rdir / "inputs.sha").write_text(_sig)
    _status = pd.read_csv(_st_file) if _st_file.is_file() else pd.DataFrame()
    if len(_status):
        show_table(_status, "R steps", index=False, csv="s10_r_status.csv")
    _py = {(m, k): RESULTS[f"{m}__{k}"] for m in MECHANISMS for k in ("missForest", "MICE-Bayes")
           if f"{m}__{k}" in RESULTS}
    R_TABLE, _ = mipy.rbridge.compare(_rdir, {m: AMPUTED[m] for m in MECHANISMS}, X, _py,
                                      seed=SEED, m=N_IMPUTATIONS)
    if len(R_TABLE):
        show_table(R_TABLE, "Table 10. Python implementations against the original R packages",
                   index=False, precision=4, csv="table10_r_agreement.csv")
        _si = R_TABLE[R_TABLE.method == "softImpute"]
        if len(_si):
            num("RSoftCellR", round(float(_si["cell r"].min()), 4), ".4f")
            num("RSoftMaxDiff", f"{float(_si['max |diff| (SD units)'].max()):.1e}")
            check("Q1", "A", "softImpute: Python and R agree cell by cell (r >= 0.995)",
                  bool((_si["cell r"] >= 0.995).all()),
                  f"lowest r = {_si['cell r'].min():.5f}")
        _lc = R_TABLE[R_TABLE.method.str.contains("imputeLCMD")]
        if len(_lc):
            num("RLcmdMaxDiff", f"{float(_lc['max |diff| (parameters)'].max()):.1e}")
            check("Q2", "A", "MinProb/QRILC: Python reproduces imputeLCMD's parameters (< 1e-6)",
                  bool((_lc["max |diff| (parameters)"] < 1e-6).all()),
                  f"largest difference {float(_lc['max |diff| (parameters)'].max()):.2e}")
        for meth, tag in (("missForest", "MF"), ("MICE-Bayes vs mice norm.boot", "Mice")):
            s = R_TABLE[R_TABLE.method == meth]
            if len(s):
                num(f"R{tag}NRMSER", round(float(s["nRMSE R"].mean()), 3), ".3f")
                num(f"R{tag}NRMSEPy", round(float(s["nRMSE Python"].mean()), 3), ".3f")
                num(f"R{tag}ProteinRho", round(float(s["protein nRMSE rho"].min()), 2), ".2f")
        _okst = set(_status[_status.status == "ok"].step) if len(_status) else set()
        MICE_RIDGE = "mice-ridge" in _okst
        MICE_NORM = "mice-norm" in _okst
        num("RMiceRidge", "yes" if MICE_RIDGE else "no")
        if MICE_RIDGE or MICE_NORM:
            print("  note: mice's norm.boot stopped (rank-deficient bootstrap samples) and was rerun "
                  + ("with its ridge option, ls.meth = \"ridge\"" if MICE_RIDGE else "")
                  + (" / " if MICE_RIDGE and MICE_NORM else "")
                  + ("with mice's Bayesian norm method" if MICE_NORM else ""))
        _vf = _rdir / "versions.csv"
        if _vf.is_file():
            for _, r in pd.read_csv(_vf).iterrows():
                num(f"RVer{re.sub('[^A-Za-z]', '', str(r['package']))}", str(r["version"]))
    else:
        print("  no R output to compare (see the R steps table)")
print("CELL 21 OK   R agreement " + ("done" if R_TABLE is not None and len(R_TABLE) else "not available"))

### CELL 22   STAGE 4.9   DATA HANDLING AT SCALE   (Table 6 of the manuscript)

In [ ]:
# ======================================================================
#  CELL 22   STAGE 4.9   DATA HANDLING AT SCALE   (Table 6 of the manuscript)
#  The steps that break conventional software on cohorts of millions of
#  patients: matching patient IDs between a visit table and a clinical
#  table, reshaping the analysis matrix to a long table and back, and
#  imputing in chunks. Simulated cohorts of BIG_SIZES patients (about 3.7
#  visits each, BIG_FEATURES proteins), with pandas, polars and DuckDB.
#  A step predicted to take longer than BIG_STEP_LIMIT seconds, or to need
#  more than 60% of the free memory, is skipped and reported as such.
#  Finished steps are saved, so a re-run continues where it stopped.
# ======================================================================
BIG_TABLE = None
if not RUN_BIGDATA:
    print("  skipped (RUN_BIGDATA = False in CELL 2)")
else:
    _bkey = f"bigdata_{mipy.runner.fingerprint(np.zeros(1), 'bigdata-v1', (SEED, BIG_FEATURES, BIG_STEP_LIMIT))}"
    _bdone = STORE_CK.load(_bkey) or {}
    _bdone = {k: v for k, v in _bdone.items() if k[0] in BIG_SIZES}

    def _save_row(key, row):
        _bdone[key] = row
        STORE_CK.save(_bkey, _bdone)

    print(f"  engines available: {', '.join(e for e in mipy.io.ENGINES if mipy.io.engine_available(e))}; "
          f"sizes {', '.join(f'{s:,}' for s in BIG_SIZES)} patients", flush=True)
    BIG_TABLE = mipy.bigdata.run_engine_benchmark(
        sizes=BIG_SIZES, n_features=BIG_FEATURES, step_time_limit=BIG_STEP_LIMIT,
        seed=SEED, done=dict(_bdone), on_row=_save_row)
    BIG_TABLE = BIG_TABLE.sort_values(["step", "engine", "patients"]).reset_index(drop=True)
    BIG_TABLE.to_csv(DIRS["results"] / "s11_bigdata_raw.csv", index=False)
    _ok = BIG_TABLE[BIG_TABLE.status == "ok"]
    TABLE11 = _ok.pivot_table(index=["step", "engine"], columns="patients", values="seconds")
    TABLE11.columns = [f"{c:,}" for c in TABLE11.columns]
    show_table(TABLE11.round(2), "Table 11. Seconds per step by number of patients "
               "(blank = skipped: too slow or too little memory)", csv="table11_bigdata.csv")
    _skip = BIG_TABLE[BIG_TABLE.status != "ok"]
    if len(_skip):
        show_table(_skip[["patients", "engine", "step", "status"]], "Steps not run",
                   index=False)
    _fail = BIG_TABLE[BIG_TABLE.status.str.startswith(("failed", "error"))]
    check("B1", "A", "data handling: every completed step passed its correctness check "
          "(IDs, row counts, round-trip sums)", len(_fail) == 0,
          "" if len(_fail) == 0 else "; ".join(f"{r.engine} {r.step} {r.patients:,}: {r.status}"
                                               for _, r in _fail.iterrows()))
    # largest cohort each engine handled for every reshaping step
    for eng in ("pandas", "polars", "duckdb"):
        s = _ok[(_ok.engine == eng) & (_ok.step != "match")]
        if len(s):
            num(f"BigMax{eng.capitalize()}", f"{int(s.patients.max()):,}")
    for step, tag in (("match", "Match"), ("long_to_wide", "Pivot"), ("wide_to_long", "Melt")):
        s = _ok[_ok.step == step]
        if len(s):
            top = s[s.patients == s.patients.max()].sort_values("seconds").iloc[0]
            num(f"Big{tag}Engine", top.engine)
            num(f"Big{tag}Patients", f"{int(top.patients):,}")
            num(f"Big{tag}Seconds", round(float(top.seconds), 1), ".1f")
    num("BigMaxPatients", f"{int(_ok.patients.max()):,}" if len(_ok) else "0")
    _imp = _ok[_ok.engine == "numpy"]
    if len(_imp):
        top = _imp[_imp.patients == _imp.patients.max()]
        for _, r in top.iterrows():
            num(f"BigImpute{r.step.split(':')[1].replace('-', '')}Seconds", round(float(r.seconds), 1), ".1f")

    # (The plot of earlier versions, fig14_bigdata, repeated Table 6 of the
    #  manuscript and was dropped in the revision of 19 October 2026.)
print("CELL 22 OK   data-handling stage " + ("done" if BIG_TABLE is not None else "skipped"))

### CELL 23   STAGE 4.10   A SECOND COHORT: CALIBRATED SYNTHETIC DATA

In [ ]:
# ======================================================================
#  CELL 23   STAGE 4.10   A SECOND COHORT: CALIBRATED SYNTHETIC DATA
#            (Table 12)
#  No second public longitudinal proteomic cohort is used here, so the
#  benchmark is repeated on a synthetic cohort whose protein means, SDs
#  and within-patient correlations are taken from GSE65622, with
#  SIM_PATIENTS patients x 4 visits x SIM_PROTEINS proteins and NO missing
#  values, so every masked cell has a known true value. The question is
#  whether the method ranking carries over. Replace this stage with a
#  real cohort when one is available (mipy.datasets / any wide table).
# ======================================================================
SIM_TABLE = None
if not RUN_SIM:
    print("  skipped (RUN_SIM = False in CELL 2)")
else:
    SIM = mipy.datasets.simulate_longitudinal(n_patients=SIM_PATIENTS, n_visits=4,
                                              n_proteins=SIM_PROTEINS, seed=SEED,
                                              calibrate_from=(X, PATIENT))
    _Xs = SIM["X"]
    _amp_s = mipy.masking.Amputer(_Xs, time=SIM["time"], rate=AMPUTE_RATE, strength=AMP_STRENGTH,
                                  min_obs=MIN_OBS_PER_PROTEIN)
    _sim_methods = [m for m in RUN_METHODS if SIM_SLOW or m not in SLOW_METHODS]
    _tasks = []
    for rep in range(1, SIM_REPEATS + 1):
        for j, mech in enumerate(MECHANISMS):
            amp = _amp_s.ampute(mech, SEED + 5000 + 1000 * rep + 101 * (j + 1))
            sch = mipy.build_variable_schema(amp["X"], MAX_PREDICTORS, MIN_COR)
            ctx = mipy.ImputeContext(seed=SEED, patient=SIM["patient"], time=SIM["time"], schema=sch,
                                     mice_iter=MICE_ITER, m=N_IMPUTATIONS, knn_k=KNN_K,
                                     max_predictors=MAX_PREDICTORS, min_cor=MIN_COR,
                                     gain_epochs=GAIN_EPOCHS, hyper_inner_iter=HYPER_INNER_ITER,
                                     gamma=MNAR_GAMMA)
            for name in _sim_methods:
                fp = mipy.runner.fingerprint(amp["X"], name + "|" + METHOD_REV.get(name, ""),
                                             _settings + ("sim-v1", SIM_PATIENTS, SIM_PROTEINS, MNAR_GAMMA))
                _tasks.append({"key": (rep, mech, name), "name": name, "X": amp["X"], "ctx": ctx,
                               "ckpt": f"sim{rep}_{mech}__{name}_{fp}", "keep_draws": True,
                               "post": partial(mipy.runner.score_record, X_true=_Xs, mask=amp["mask"],
                                               patient=SIM["patient"])})
    print(f"  synthetic cohort: {SIM_PATIENTS} patients x 4 visits x {_Xs.shape[1]} proteins; "
          f"{len(_tasks)} runs", flush=True)
    _res23 = mipy.runner.run_tasks(_tasks, STORE_CK, n_workers=N_WORKERS)
    SIM_TABLE = pd.DataFrame([{"repeat": k[0], "mechanism": k[1], "method": k[2], **v}
                              for k, v in _res23.items() if v.get("ok")])
    SIM_TABLE.to_csv(DIRS["results"] / "s12_sim_raw.csv", index=False)
    TABLE12 = (SIM_TABLE.groupby(["method", "mechanism"])["NRMSE"].mean().unstack()
               .reindex(columns=list(MECHANISMS)))
    TABLE12 = TABLE12.sort_values(TABLE12.columns[0])
    show_table(TABLE12.round(3), f"Table 12. Synthetic cohort: median NRMSE "
               f"(mean of {SIM_REPEATS} maskings)", gradient=list(MECHANISMS),
               csv="table12_synthetic.csv")
    _rhos = []
    for mech in MECHANISMS:
        a = TABLE12[mech]
        b = PIVOT[mech].reindex(a.index)
        g = a.notna() & b.notna()
        rho = float(_stats.spearmanr(a[g], b[g])[0])
        _rhos.append(rho)
        num(f"SimBest{mech}", a.idxmin())
        num(f"SimBest{mech}NRMSE", round(float(a.min()), 3), ".3f")
        num(f"SimRankRho{mech}", round(rho, 2), ".2f")
        print(f"  {mech}: ranking agreement with GSE65622, Spearman rho = {rho:.2f}")
    num("SimPatients", SIM_PATIENTS)
    num("SimProteins", _Xs.shape[1])
    num("SimRepeats", SIM_REPEATS)
print("CELL 23 OK   synthetic cohort " + ("done" if SIM_TABLE is not None else "skipped"))

### CELL 24   STAGE 5   SURVIVAL   (Table 13, Figure 15)

In [ ]:
# ======================================================================
#  CELL 24   STAGE 5   SURVIVAL   (Table 13, Figure 15)
#  Does the imputation method change which baseline proteins look
#  prognostic? For every method's imputation of the real data: a
#  univariate Cox model per protein on the baseline sample, pooled over
#  the m completions with Rubin's rules for multiple-imputation methods,
#  Benjamini-Hochberg false-discovery control, and agreement of the top-20
#  proteins with an available-case analysis (no imputation).
#
#  SURVIVAL_MODE (CELL 2):
#    "off"        not run (default since the revision of 19 October 2026)
#    "reconciled" any relapse or metastasis, earliest time; patients whose
#                 earliest time is zero or less are excluded. Exploratory:
#                 the RFS/MFS columns fail checks E3-E5 (CELL 8), and this
#                 endpoint is consistent by construction, not verified.
#    "gated"      RFS and MFS separately, only if gate B passed.
#    "MFS_only"   MFS alone, patients with time > 0 (exploratory).
# ======================================================================
SURV_TABLE = None
_label = "withheld"
_eps = {}
if SURVIVAL_MODE == "off":
    _label = "not run (survival is off)"
    print("  not run: SURVIVAL_MODE = 'off' in CELL 2 (the manuscript revision of "
          "19 October 2026 leaves survival out)")
elif ENDPOINTS is None:
    print("  WITHHELD. The endpoint table from CELL 8 is missing.")
elif SURVIVAL_MODE == "reconciled":
    _rec = mipy.survival.reconcile(ENDPOINTS)
    _eps["PFS (reconciled)"] = _rec.loc[~_rec.excluded, ["T", "E"]]
    _label = "reconciled progression endpoint (exploratory)"
    print(f"  reconciled endpoint: {int((~_rec.excluded).sum())} patients, "
          f"{int(_rec.loc[~_rec.excluded, 'E'].sum())} events; "
          f"{int(_rec.excluded.sum())} excluded (event before the first sample)")
    num("SurvPatients", int((~_rec.excluded).sum()))
    num("SurvEvents", int(_rec.loc[~_rec.excluded, "E"].sum()))
    num("SurvExcluded", int(_rec.excluded.sum()))
elif SURVIVAL_MODE == "gated" and GATE_B_OK:
    for ep in ("RFS", "MFS"):
        d = ENDPOINTS[[f"t_{ep}", f"e_{ep}"]].set_axis(["T", "E"], axis=1)
        _eps[ep] = d[d["T"] > 0]
    _label = "RFS and MFS"
elif SURVIVAL_MODE == "MFS_only":
    d = ENDPOINTS[["t_MFS", "e_MFS"]].set_axis(["T", "E"], axis=1)
    _eps["MFS"] = d[d["T"] > 0]
    _label = "MFS only (exploratory)"
else:
    print("  WITHHELD. Gate B failed in CELL 8 and SURVIVAL_MODE is 'gated'.")

if _eps:
    _first = pd.Series(np.arange(len(PATIENT))).groupby(PATIENT).first()   # baseline rows
    _rows, _tops = [], {}
    for ep, sv in _eps.items():
        ids = [i for i in _first.index if i in sv.index]
        rows_b = _first.loc[ids].to_numpy()
        T, E = sv.loc[ids, "T"].to_numpy(dtype=float), sv.loc[ids, "E"].to_numpy(dtype=int)
        # available-case reference: each protein uses only its observed baseline values
        Xb = X[rows_b]
        ref = []
        for k in range(Xb.shape[1]):
            ok = np.isfinite(Xb[:, k])
            if ok.sum() >= 20 and E[ok].sum() >= 3:
                r = mipy.survival.univariate_cox(T[ok], E[ok], Xb[ok, k:k + 1]).iloc[0]
                ref.append({"protein": PROTEINS[k], "log_hr": r.beta, "p": r.p})
            else:
                ref.append({"protein": PROTEINS[k], "log_hr": np.nan, "p": np.nan})
        ref = pd.DataFrame(ref)
        ref["q"] = mipy.survival.bh_fdr(ref["p"].to_numpy())
        top_ref = set(ref.nsmallest(20, "p").protein)
        _tops[(ep, "available-case")] = top_ref
        _rows.append({"endpoint": ep, "method": "available-case (no imputation)", "m": 1,
                      "patients": len(ids), "events": int(E.sum()),
                      f"proteins q < {SURV_FDR}": int((ref.q < SURV_FDR).sum()),
                      "top-20 overlap with available-case": 20, "log-HR r with available-case": 1.0})
        for name, rec in RESULTS_REAL.items():
            if not rec.get("ok"):
                continue
            comps = rec["draws"] if rec.get("draws") else [rec["imputed"]]
            res = mipy.survival.pooled_univariate_cox(T, E, [C[rows_b] for C in comps], PROTEINS)
            top = set(res.nsmallest(20, "p").protein)
            _tops[(ep, name)] = top
            g = ref.log_hr.notna()
            _rows.append({"endpoint": ep, "method": name, "m": len(comps), "patients": len(ids),
                          "events": int(E.sum()),
                          f"proteins q < {SURV_FDR}": int((res.q < SURV_FDR).sum()),
                          "top-20 overlap with available-case": len(top & top_ref),
                          "log-HR r with available-case":
                              float(np.corrcoef(res.log_hr[g], ref.log_hr[g])[0, 1])})
            res.to_csv(DIRS["results"] / f"s13_cox_{ep.split()[0]}_{name}.csv", index=False)
    SURV_TABLE = pd.DataFrame(_rows)
    show_table(SURV_TABLE, f"Table 13. Univariate Cox screen of baseline proteins by imputation "
               f"method ({_label})", index=False, csv="table13_survival.csv")
    _ep0 = list(_eps)[0]
    s = SURV_TABLE[(SURV_TABLE.endpoint == _ep0) & (SURV_TABLE.method != "available-case (no imputation)")]
    num("SurvSigMin", int(s[f"proteins q < {SURV_FDR}"].min()))
    num("SurvSigMax", int(s[f"proteins q < {SURV_FDR}"].max()))
    num("SurvOverlapMin", int(s["top-20 overlap with available-case"].min()))
    num("SurvOverlapMax", int(s["top-20 overlap with available-case"].max()))
    num("SurvFDR", SURV_FDR, ".2f")
    # Jaccard overlap of the top-20 lists between methods
    _ms = ["available-case"] + [m for m in RESULTS_REAL if (_ep0, m) in _tops]
    J = pd.DataFrame([[len(_tops[(_ep0, a)] & _tops[(_ep0, b)]) / len(_tops[(_ep0, a)] | _tops[(_ep0, b)])
                       for b in _ms] for a in _ms], index=_ms, columns=_ms)
    J.to_csv(DIRS["results"] / "s13_top20_jaccard.csv")
    _jo = J.where(~np.eye(len(J), dtype=bool)).stack()
    num("SurvJaccardMedian", round(float(_jo.median()), 2), ".2f")

    fig, ax = plt.subplots(figsize=(0.42 * len(_ms) + 2.2, 0.42 * len(_ms) + 1.6))
    im = ax.imshow(J.to_numpy(), cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(_ms)))
    ax.set_xticklabels(_ms, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(len(_ms)))
    ax.set_yticklabels(_ms, fontsize=7)
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.7, label="Jaccard overlap of top-20 proteins")
    show_fig(fig, "fig15_survival_overlap")
num("SurvivalRun", _label)
print(f"CELL 24 OK   survival stage: {_label}")

### CELL 25   STAGE 6.1   EVIDENCE TABLE AND DECISION GUIDE

In [ ]:
# ======================================================================
#  CELL 25   STAGE 6.1   EVIDENCE TABLE AND DECISION GUIDE
#  (Tables 14 and 15, Figure 16)
#  Table 14: the Python imputation packages and the peer-reviewed
#            evidence behind each (checked against the papers/pages).
#  Table 15 + Figure 16: which method to use in which situation, chosen
#            from this run's results by fixed rules (written below), so
#            the guide changes if the results do.
# ======================================================================
LIT = mipy.literature()
show_table(LIT[["package", "methods", "package_paper", "peer_reviewed", "longitudinal",
                "high_dimensional"]], "Table 14. Python imputation packages and their evidence",
           index=False, csv="table14_literature.csv")
num("LitNPackages", len(LIT) - 1)
num("LitNNoPaper", int(LIT.peer_reviewed.str.startswith("no").sum()))
num("LitNLongitudinal", int(LIT.longitudinal.str.startswith("yes").sum()) - 1)

_acc = SUMMARY.pivot(index="method", columns="mechanism", values="NRMSE_median")
_phi = SUMMARY.pivot(index="method", columns="mechanism", values="structure_ratio")
_guide = []

# 1. random missingness: best mean MCAR/MAR accuracy (revision of 19 October
#    2026: the earlier filter on the SD ratio of the mean imputation is gone,
#    because the multiple-imputation methods keep the ratio per completion,
#    see rule 5)
_rand = [m for m in ("MCAR", "MAR") if m in _acc.columns]
_s = _acc[_rand].mean(axis=1).sort_values()
_guide.append({"situation": "Missing at random, moderate size",
               "rule": "lowest mean nRMSE (MCAR, MAR)",
               "recommended": _s.index[0], "runner-up": _s.index[1] if len(_s) > 1 else "",
               "evidence": f"nRMSE {_s.iloc[0]:.3f}"})
num("GuideAnRMSE", round(float(_s.iloc[0]), 3), ".3f")
if len(_s) > 1:
    num("GuideARunnerUpnRMSE", round(float(_s.iloc[1]), 3), ".3f")
# 2. detection limit suspected: best mean over the MNAR designs, plus a
#    sensitivity analysis over gamma
_mn = TABLE8[[c for c in TABLE8.columns if c.startswith("MNAR") or c == "LOD"]].mean(axis=1).sort_values()
_guide.append({"situation": "Values lost below a detection limit (MNAR)",
               "rule": "lowest mean nRMSE over the MNAR and LOD designs; report the gamma grid",
               "recommended": _mn.index[0], "runner-up": _mn.index[1] if len(_mn) > 1 else "",
               "evidence": f"nRMSE {_mn.iloc[0]:.3f}"})
num("GuideBnRMSE", round(float(_mn.iloc[0]), 3), ".3f")
num("GuideBNDesigns", len([c for c in TABLE8.columns if c.startswith("MNAR") or c == "LOD"]))
# 3. uncertainty needed: multiple imputation with calibrated intervals
if TABLE4 is not None:
    _mc = TABLE4[TABLE4.mechanism == "MCAR"].set_index("method")
    _cal = _mc[(_mc["coverage of 95% intervals"] - 0.95).abs() <= 0.05]
    _cal = (_cal if len(_cal) else _mc).sort_values("nRMSE (mean of draws)")
    _guide.append({"situation": "Uncertainty must be carried forward (multiple imputation)",
                   "rule": "lowest nRMSE among MI methods whose 95% intervals cover 90-100% under MCAR",
                   "recommended": _cal.index[0], "runner-up": _cal.index[1] if len(_cal) > 1 else "",
                   "evidence": f"coverage {100 * _cal.iloc[0]['coverage of 95% intervals']:.0f}%"})
    num("GuideCCoverage", round(100 * float(_cal.iloc[0]["coverage of 95% intervals"]), 1), ".1f")
# 4. large cohorts: most accurate method that finished at the largest
#    simulated size within the time limit
if SCALING is not None:
    _big = SCALING[(SCALING.rows == SCALING.rows.max()) & (SCALING.status == "ok")].method
    _sb = _acc.loc[[m for m in _big if m in _acc.index], _rand].mean(axis=1).sort_values()
    if len(_sb):
        _guide.append({"situation": f"Large cohorts ({int(SCALING.rows.max()):,} rows and more)",
                       "rule": f"lowest nRMSE among methods finishing within {SCALE_TIME_LIMIT} s "
                               f"at {int(SCALING.rows.max()):,} rows",
                       "recommended": _sb.index[0], "runner-up": _sb.index[1] if len(_sb) > 1 else "",
                       "evidence": f"nRMSE {_sb.iloc[0]:.3f}"})
        num("GuideDnRMSE", round(float(_sb.iloc[0]), 3), ".3f")
        num("NFinishLargest", len(_sb))
        num("FinishLargestMethods", ", ".join(sorted(_sb.index, key=str.lower)))
# 5. trajectories matter: the per-completion within-patient SD ratio (MCAR,
#    MAR) closest to 1 among the multiple-imputation methods; each
#    completion is analysed and the results pooled by Rubin's rules
#    (revision of 19 October 2026; before, the SD ratio of the mean
#    imputation among the five most accurate methods was used)
if TABLE4 is not None:
    _sr = (TABLE4[TABLE4.mechanism.isin(_rand)]
           .pivot(index="method", columns="mechanism", values="SD ratio per completion"))
    _tr = _sr[_rand].sub(1).abs().mean(axis=1).sort_values()
    _guide.append({"situation": "Within-patient change is the outcome",
                   "rule": "per-completion SD ratio (MCAR, MAR) closest to 1 among MI methods; "
                           "completions pooled by Rubin's rules",
                   "recommended": _tr.index[0], "runner-up": _tr.index[1] if len(_tr) > 1 else "",
                   "evidence": f"mean |SD ratio - 1| = {_tr.iloc[0]:.3f}"})
    num("GuideEDev", round(float(_tr.iloc[0]), 3), ".3f")
TABLE15 = pd.DataFrame(_guide)
show_table(TABLE15, "Table 15. Which method in which situation (rules applied to this run)",
           index=False, csv="table15_decision.csv")
_GKEY = {"Missing at random, moderate size": "A",
         "Values lost below a detection limit (MNAR)": "B",
         "Uncertainty must be carried forward (multiple imputation)": "C",
         "Within-patient change is the outcome": "E"}
for i, r in TABLE15.iterrows():
    k = _GKEY.get(r.situation, "D")          # D = large cohorts
    num(f"Guide{k}", r.recommended)
    num(f"Guide{k}RunnerUpName", r["runner-up"])

# ---- Figure 16: the decision guide as a flowchart (TikZ, for the manuscript) ------
def _t(s):
    return texsafe(s)


_g = {r.situation: r for _, r in TABLE15.iterrows()}
_lines = [r"% Generated by the MIPY notebook (CELL 25). Do not edit by hand.",
          r"\begin{tikzpicture}[font=\footnotesize, node distance=4mm and 6mm,",
          r"  q/.style={draw=black!60, rounded corners=2pt, fill=black!4, align=center, text width=3.6cm, minimum height=9mm},",
          r"  a/.style={draw=#1, line width=0.8pt, rounded corners=2pt, fill=#1!10, align=center, text width=3.0cm, minimum height=9mm},",
          r"  ar/.style={-{Stealth[length=2mm]}, draw=black!55}]"]
_cols = ["0072B2", "E69F00", "009E73", "CC79A7", "56B4E9"]
for i, r in enumerate(TABLE15.itertuples()):
    _lines.append(rf"\definecolor{{g{i}}}{{HTML}}{{{_cols[i % 5]}}}")
    pos = "" if i == 0 else rf", below=of q{i - 1}"
    _lines.append(rf"\node[q{pos}] (q{i}) {{{_t(r.situation)}?}};")
    _lines.append(rf"\node[a=g{i}, right=of q{i}] (a{i}) {{\textbf{{{_t(r.recommended)}}}\\[-1pt]"
                  rf"\scriptsize {_t(r.evidence)}}};")
    _lines.append(rf"\draw[ar] (q{i}) -- node[above, font=\scriptsize]{{yes}} (a{i});")
    if i:
        _lines.append(rf"\draw[ar] (q{i - 1}) -- node[left, font=\scriptsize]{{no}} (q{i});")
_lines.append(r"\end{tikzpicture}")
(DIRS["manuscript"] / "decision_guide.tex").write_text("\n".join(_lines), encoding="utf-8")
print("CELL 25 OK   tables 14-15; flowchart written to manuscript/decision_guide.tex")

### CELL 26   STAGE 6.2   tables.tex, numbers.tex, supplementary.tex,

In [ ]:
# ======================================================================
#  CELL 26   STAGE 6.2   tables.tex, numbers.tex, supplementary.tex,
#                        PROVENANCE AND THE FINAL CHECK LIST
#  Writes the files the manuscript reads (mipy_run\manuscript\). Copy
#  numbers.tex, tables.tex, supplementary.tex, decision_guide.tex and
#  figures\*.pdf into Overleaf and recompile.
# ======================================================================
def _latex_table(label, caption, body):
    return ("\\begin{table}[htbp]\n\\centering\n\\small\n"
            f"\\caption{{{caption}}}\n\\label{{{label}}}\n"
            "\\resizebox{\\textwidth}{!}{%\n" + body + "\n}\n\\end{table}\n\n")


def _dec(k):
    return lambda v: "--" if pd.isna(v) else f"{v:.{k}f}"


def _tl(df, **kw):
    """DataFrame -> LaTeX tabular with escaped text."""
    kw.setdefault("na_rep", "--")
    return df.to_latex(**kw)


_tex = []
_t1 = TABLE1[TABLE1.status == "will run"].drop(columns="status")
_t1.columns = ["Method", "Family", "Package", "Function", "Peer-reviewed", "MI", "New in MIPY"]
_tex.append(_latex_table("tab:methods",
    "Imputation methods benchmarked, with the package and function actually invoked, whether a "
    "peer-reviewed publication describes the method as implemented (yes; adapted = a published "
    "method changed in engine, stopping rule or unit of application; no), whether it returns "
    "multiple imputations, and whether it is introduced in this work.",
    _tl(_t1, index=False, escape=True)))
_tex.append(_latex_table("tab:nrmse",
    "Median per-protein nRMSE by method and missingness mechanism. A value of 1.00 indicates "
    "performance indistinguishable from imputing each protein by its own mean.",
    _tl(_t2.round(3), escape=True, float_format="%.3f")))
_t3x = _t3.copy()
_t3x.columns = ["Mechanism", "Method", "nRMSE", "r", "MAE", "Bias", "SD ratio",
                "\\% nRMSE $<$ 1", "s"]
_tex.append(_latex_table("tab:full",
    "Complete evaluation. nRMSE, correlation, MAE and bias are computed within each protein and "
    "summarised by the median across proteins. SD ratio is the pooled within-patient standard "
    "deviation of the completed data divided by that of the true values on the same cells "
    "(1 = preserved). A dash marks a correlation that is undefined because the method imputes a "
    "constant within each protein. s = run time in seconds.",
    _tl(_t3x, index=False, escape=False, formatters={
        **{c: _dec(3) for c in ["nRMSE", "r", "MAE", "Bias", "SD ratio"]},
        "\\% nRMSE $<$ 1": _dec(1), "s": _dec(1)})))
if TABLE4 is not None:
    _t4x = TABLE4[["mechanism", "method", "m", "nRMSE (mean of draws)", "FMI, imputed cells",
                   "median FMI, protein means", "coverage of 95% intervals", "SD ratio per completion"]].copy()
    _t4x["coverage of 95% intervals"] *= 100
    _t4x.columns = ["Mechanism", "Method", "$m$", "nRMSE", "FMI (cells)", "FMI (proteins)",
                    "Coverage (\\%)", "SD ratio"]
    _tex.append(_latex_table("tab:rubin",
        "Multiple imputation. FMI = fraction of missing information by Rubin's rules, for the mean "
        "of the imputed cells and as the median over protein means. Coverage = share of masked "
        "cells whose true value lies in the 95\\% interval formed from the $m$ completions "
        "(nominal 95\\%). SD ratio is averaged over completions.",
        _tl(_t4x, index=False, escape=False, formatters={
            "nRMSE": _dec(3), "FMI (cells)": _dec(3), "FMI (proteins)": _dec(3),
            "Coverage (\\%)": _dec(1), "SD ratio": _dec(3)})))
if "TABLE6" in globals() and len(TABLE6):
    _t6x = TABLE6[["mechanism", "method", "repeats", "NRMSE mean", "MCSE", "NRMSE 2.5%",
                   "NRMSE 97.5%", "SD ratio mean", "mean rank", "% best"]].copy()
    _t6x.columns = ["Mechanism", "Method", "Repeats", "nRMSE", "MCSE", "2.5\\%", "97.5\\%",
                    "SD ratio", "Mean rank", "\\% best"]
    _tex.append(_latex_table("tab:repeats",
        f"Stability over up to {N_REPEATS} independent maskings: mean of the median per-protein "
        "nRMSE with its Monte Carlo standard error and central 95\\% range. Rank and \\% best use "
        "the maskings that every method completed.",
        _tl(_t6x, index=False, escape=False, formatters={
            **{c: _dec(3) for c in ["nRMSE", "2.5\\%", "97.5\\%", "SD ratio"]},
            "MCSE": _dec(4), "Mean rank": _dec(1), "\\% best": _dec(0)})))
if "SCALING" in globals() and SCALING is not None:
    _t7x = TABLE7.copy()
    _t7x.columns = [c.replace(" rows (s)", "") for c in _t7x.columns]
    _tex.append(_latex_table("tab:scaling",
        f"Run time in seconds on simulated data with {SCALING_P} proteins and 10\\% of cells "
        "missing completely at random, by number of rows. A dash marks a size not attempted "
        f"because the predicted time exceeded {SCALE_TIME_LIMIT} s.",
        _tl(_t7x, escape=True, float_format="%.2f")))
_tex.append(_latex_table("tab:sensitivity",
    "Median per-protein nRMSE as missingness becomes more strongly not at random: MNAR masking "
    "with logistic slopes $\\beta$"
    + (f", a probit selection with slope {PROBIT_BETA:.2f}" if PROBIT_BETA is not None else "")
    + ", and a hard detection limit (LOD). The four slowest methods are not included.",
    _tl(TABLE8.round(3), escape=True, float_format="%.3f")))
if "ESS_TABLE" in globals() and ESS_TABLE is not None:
    _tess = ESS_TABLE[["design", "gamma", "median ESS", "10th percentile", "minimum",
                       "% cells with ESS < 10"]].copy()
    _tess.columns = ["Design", "$\\gamma$", "Median ESS", "10th pct.", "Min.", "ESS $<$ 10 (\\%)"]
    _tex.append(_latex_table("tab:ess",
        "MICE-LongMNAR: effective sample size $(\\sum_j w_{ikj})^2/\\sum_j w_{ikj}^2$ of the "
        "importance resampling over $C=50$ candidates, over the imputed cells of the last sweep "
        f"of each of the {N_IMPUTATIONS} completions.",
        _tl(_tess, index=False, escape=False, formatters={
            "Median ESS": _dec(1), "10th pct.": _dec(1), "Min.": _dec(1),
            "ESS $<$ 10 (\\%)": _dec(2), "$\\gamma$": _dec(1)})))
_tex.append(_latex_table("tab:realmissing",
    "Where each method places the genuinely missing values of GSE65622: median standardised "
    "shift of the imputed values from the observed mean, share of proteins shifted downwards, "
    "and median share of imputed values below the lowest observed value.",
    _tl(TABLE9, index=False, escape=True, float_format="%.2f")))
if R_TABLE is not None and len(R_TABLE):
    _t10 = R_TABLE.copy()
    _tex.append(_latex_table("tab:ragreement",
        "MIPY's Python implementations against the original R packages on the same masked cells. "
        "softImpute uses the same standardisation and penalty; imputeLCMD parameters are "
        "deterministic; missForest and mice use different engines, so accuracy is compared."
        + " Protein rho is the Spearman correlation of per-protein nRMSE between R and Python."
        + (" mice's norm.boot stopped on rank-deficient bootstrap samples, so the mice rows marked "
           "in the comparison column use norm.boot with mice's ridge option (ls.meth = \"ridge\")"
           + (" or mice's norm method" if MICE_NORM else "") + "." if (MICE_RIDGE or MICE_NORM) else ""),
        _tl(_t10, index=False, escape=True, float_format="%.4g")))
if BIG_TABLE is not None:
    _tex.append(_latex_table("tab:bigdata",
        f"Seconds per data-handling step by cohort size (patients, about 3.7 visits each, "
        f"{BIG_FEATURES} proteins). A dash marks a step skipped because it was predicted to "
        f"exceed {BIG_STEP_LIMIT} s or the available memory.",
        _tl(TABLE11.round(2), escape=True, float_format="%.2f")))
if SIM_TABLE is not None:
    _tex.append(_latex_table("tab:synthetic",
        f"Synthetic cohort calibrated to GSE65622 ({SIM_PATIENTS} patients, 4 visits, "
        f"{SIM_PROTEINS} proteins, no missing values): median per-protein nRMSE, mean of "
        f"{SIM_REPEATS} maskings.", _tl(TABLE12.round(3), escape=True, float_format="%.3f")))
if SURV_TABLE is not None:
    _tex.append(_latex_table("tab:survival",
        f"Univariate Cox screen of baseline proteins by imputation method ({texsafe(_label)}). "
        f"Multiple-imputation methods are pooled with Rubin's rules. q = Benjamini-Hochberg "
        f"adjusted p-value.", _tl(SURV_TABLE, index=False, escape=True, float_format="%.2f")))
_t14 = LIT[["package", "package_paper", "peer_reviewed", "longitudinal", "high_dimensional"]].copy()
_t14.columns = ["Package", "Publication", "Peer-reviewed", "Longitudinal evaluation", "High-dimensional"]
_tex.append(_latex_table("tab:literature",
    "Python imputation packages and the peer-reviewed evidence behind them (checked October 2026).",
    _tl(_t14, index=False, escape=True)))
_t15 = TABLE15[["situation", "recommended", "runner-up", "rule", "evidence"]].copy()
_t15.columns = ["Situation", "Recommended", "Runner-up", "Rule applied to this run", "Evidence"]
_tex.append(_latex_table("tab:decision", "Which method in which situation.",
                         _tl(_t15, index=False, escape=True)))

with open(DIRS["manuscript"] / "tables.tex", "w", encoding="utf-8") as f:
    f.write("% Generated by the MIPY notebook (v2). Do not edit by hand.\n")
    f.write(f"% {datetime.now().isoformat(timespec='seconds')}\n\n")
    f.writelines(_tex)

# ---- numbers.tex ------------------------------------------------------------
num("NFigures", len(list(DIRS["figures"].glob("fig*.pdf"))))
num("NTables", len(_tex))
num("NChecksPassed", sum(c["passed"] for c in CHECKS))
num("NChecksTotal", len(CHECKS))
_rep_sec = (float(REPEATS.loc[REPEATS["repeat"] > 1, "seconds"].sum())
            if "REPEATS" in globals() and len(REPEATS) else 0.0)
num("ComputeHoursMain", round(sum(r["seconds"] for r in RESULTS.values()) / 3600, 2), ".2f")
num("ComputeHoursRepeats", round(_rep_sec / 3600, 2), ".2f")
num("ComputeHoursReal", round(sum(r["seconds"] for r in RESULTS_REAL.values()) / 3600, 2), ".2f")
num("Seed", SEED)
num("NWorkers", N_WORKERS)
num("Profile", PROFILE)
for _k in ("python", "numpy", "pandas", "sklearn", "lightgbm", "miceforest",
           "hyperimpute", "torch", "lifelines", "polars", "duckdb", "mipy"):
    num("Ver" + _k.capitalize(), ENV.get(_k) or "not installed")

_macros = {}
for key in NUMBERS:
    _macros.setdefault(re.sub(r"[^A-Za-z]", "", key), []).append(key)
_clash = {m: ks for m, ks in _macros.items() if len(ks) > 1}
if _clash:
    print("  WARNING: these numbers share a LaTeX name; only the first is written:",
          "; ".join(f"{m}: {ks}" for m, ks in _clash.items()))
with open(DIRS["manuscript"] / "numbers.tex", "w", encoding="utf-8") as f:
    f.write("% Generated by the MIPY notebook (v2). Do not edit by hand.\n")
    f.write("% Every number the manuscript quotes is defined here.\n")
    f.write(f"% {datetime.now().isoformat(timespec='seconds')}\n\n")
    _written = set()
    for key, d in sorted(NUMBERS.items()):
        name = re.sub(r"[^A-Za-z]", "", key)
        if name in _written:
            continue
        _written.add(name)
        v, fmt = d["value"], d["fmt"]
        if v is None or (isinstance(v, float) and not np.isfinite(v)):
            s = "n/a"
        elif fmt and isinstance(v, (int, float, np.integer, np.floating)):
            s = format(v, fmt)
        else:
            s = str(v)
        if not d["raw"]:
            s = texsafe(s)
        f.write(f"\\newcommand{{\\{name}}}{{{s}}}\n")

# ---- supplementary.tex: environment, settings, hardware ----------------------
try:
    import psutil as _ps
    _ram = f"{_ps.virtual_memory().total / 2 ** 30:.1f} GB"
except Exception:
    _ram = "unknown"
_hw = {"platform": ENV.get("platform"), "processor": platform.processor() or platform.machine(),
       "logical CPUs": os.cpu_count(), "memory": _ram, "worker processes": N_WORKERS,
       "profile": PROFILE, "seed": SEED}
_set = {"log transform": LOG_TRANSFORM, "masking rate": AMPUTE_RATE, "masking slope": AMP_STRENGTH,
        "minimum observed per protein": MIN_OBS_PER_PROTEIN, "quickpred predictors": MAX_PREDICTORS,
        "quickpred threshold": MIN_COR, "chained-equation sweeps": MICE_ITER,
        "completions m": N_IMPUTATIONS, "KNN k": KNN_K,
        "GAIN training iterations (used)": (GAIN_USED or {}).get("n_epochs", "not recorded"),
        "GAIN batch size, hint rate (used)": (f"{GAIN_USED['batch_size']}, {GAIN_USED['hint_rate']}"
                                              if GAIN_USED else "not recorded"),
        "HyperImpute inner iterations": HYPER_INNER_ITER, "MICE-LongMNAR gamma": MNAR_GAMMA,
        "maskings": N_REPEATS, "maskings, slow methods": N_REPEATS_SLOW if REPEAT_SLOW else 1,
        "probit masking slope": round(PROBIT_BETA, 3) if PROBIT_BETA is not None else "not run"}
if SURVIVAL_MODE != "off":
    _set["survival mode"] = SURVIVAL_MODE
_vers = {k: ENV.get(k) for k in ("python", "numpy", "pandas", "scipy", "sklearn", "matplotlib",
                                 "lightgbm", "miceforest", "hyperimpute", "torch", "lifelines",
                                 "polars", "duckdb", "psutil", "mipy")}
_vers["R"] = ENV.get("R") and "found"
_sup = ["% Generated by the MIPY notebook (v2). Do not edit by hand."]
for title, d in (("Computing environment", _hw), ("Analysis settings", _set), ("Package versions", _vers)):
    _df = pd.DataFrame([(k, "--" if v is None else str(v)) for k, v in d.items()], columns=["Item", "Value"])
    _sup.append(_latex_table(f"tab:sup{title.split()[0].lower()}", title, _tl(_df, index=False, escape=True)))
(DIRS["manuscript"] / "supplementary.tex").write_text("\n".join(_sup), encoding="utf-8")

# ---- provenance -----------------------------------------------------------------
(DIRS["provenance"] / "checks.json").write_text(json.dumps(CHECKS, indent=2), encoding="utf-8")
(DIRS["provenance"] / "environment.json").write_text(json.dumps({**ENV, **_hw}, indent=2, default=str),
                                                     encoding="utf-8")
with open(DIRS["provenance"] / "numbers.json", "w", encoding="utf-8") as f:
    json.dump({k: v["value"] for k, v in NUMBERS.items()}, f, indent=2, default=str)

show_table(pd.DataFrame(CHECKS)[["id", "gate", "label", "passed"]], "All checks", index=False)
_files = sorted(p.relative_to(OUT_DIR).as_posix()
                for p in OUT_DIR.rglob("*") if p.is_file() and p.parent.name != "ckpt")
_failA = [c["id"] for c in CHECKS if c["gate"] == "A" and not c["passed"]]
print(f"\n  {len(NUMBERS)} numbers written to manuscript/numbers.tex")
print(f"  {len(_tex)} tables written to manuscript/tables.tex (+ supplementary.tex, decision_guide.tex)")
print(f"  {len(_files)} output files under {OUT_DIR}")
_stale = sorted(p.relative_to(OUT_DIR).as_posix() for d in ("figures", "results")
                for p in (OUT_DIR / d).glob("*") if p.is_file() and p.stat().st_mtime < T0 - 1)
if _stale:
    print(f"  {len(_stale)} older files in figures/ and results/ were not written by this run "
          f"(left from an earlier version; the v2 manuscript does not use them):")
    for f in _stale[:12]:
        print(f"      {f}")
    if len(_stale) > 12:
        print(f"      ... and {len(_stale) - 12} more")
print(f"  checks: {sum(c['passed'] for c in CHECKS)} of {len(CHECKS)} passed")
print(f"  gate A: {'all passed' if not _failA else 'FAILED ' + ', '.join(_failA)}"
      + ("   gate B: not run (survival off)" if SURVIVAL_MODE == "off" else
         f"   gate B: {'passed' if GATE_B_OK else 'failed (survival: ' + _label + ')'}"))
print(f"  total time this session: {(time.time() - T0) / 60:.1f} minutes")
print("\nCELL 26 OK   copy manuscript/*.tex and figures/*.pdf into Overleaf, then recompile.")